# Sesión 7 · RAG avanzado: técnicas, GraphRAG y evaluación

**AI Engineer I: Fundamentos de IA Generativa** · Módulo 1 · Edutek

---

## 🎯 Objetivos de la sesión

Al terminar este notebook podrás:

1. **Explicar por qué falla un RAG básico** con preguntas que necesitan varios fragmentos (multi-sección) o varios pasos de razonamiento (multi-salto), y **medirlo**.
2. **Aplicar técnicas avanzadas de recuperación**: *small-to-big* (buscar con lo pequeño, entregar lo grande), encabezados y *contextual retrieval*, y descomposición de preguntas (con HyDE como lectura).
3. **Explicar GraphRAG**: qué es un grafo de conocimiento, cuándo aporta (preguntas relacionales y globales) y cuándo **no** compensa; y compararlo **con números** frente a la búsqueda vectorial.
4. **Evaluar un RAG por capas**: Recall@K, Precision@K y MRR para la recuperación; dato correcto, citas válidas, "no sé" y cifras con respaldo para la generación; un **juez LLM** calibrado contra etiquetas humanas.
5. **Reconocer las causas de las alucinaciones, mitigarlas y comprobar la mejora** con el ciclo *medir → cambiar → volver a medir*.
6. **Ubicar los recuperadores modernos** (dispersos aprendidos como SPLADE, de interacción tardía como ColBERT/ColPali) y **optimizar embeddings** con Matryoshka y cuantización, midiendo el efecto en la recuperación.
7. **Construir un RAG multimodal de audio y video**: transcribir con Whisper, fragmentar **con tiempo**, preguntar en español sobre contenido en inglés y responder con citas `mm:ss`.
8. **Llevar un RAG a producción**: caché semántica y de prompt, actualización incremental del índice, ciclo de vida y observabilidad.

> 🔁 **Lo que NO repetimos.** La reescritura de la consulta, la búsqueda híbrida (palabras clave + semántica) y el reranking son de la Sesión 6. Aquí los damos por sabidos y nos centramos en lo que viene después de ellos.

> ⚠️ **Sigue sin ser un agente.** Todo lo de hoy son flujos fijos. El *RAG agéntico* (donde el modelo decide qué buscar y cuándo parar) se verá en sesiones posteriores.

## 🗺️ Mapa del notebook

| Parte | Tema | ¿Consume dinero? |
| --- | --- | --- |
| 0 | Preparar el entorno | No |
| 1 | Medir antes de mejorar: golden set y métricas de recuperación | No |
| 2 | Dónde falla el RAG básico | No |
| 3 | *Small-to-big*, encabezados contextuales y *contextual retrieval* | Solo *contextual retrieval*: centavos (gratis con modelo local) |
| 4 | Descomposición de preguntas (HyDE, de lectura) | Opcional: centavos (gratis con modelo local) |
| 5 | **GraphRAG**: qué es y cuándo conviene | No |
| 6 | GraphRAG medido: vectorial vs grafo vs combinado | Opcional: centavos (gratis con modelo local) |
| 7 | **Modelos de recuperación modernos y embeddings**: Matryoshka y cuantización | No |
| 8 | **RAG multimodal**: audio y video (requiere `--group media`) | Casi no: 3 respuestas (gratis con modelo local) |
| 9 | Evaluar la generación y diagnosticar por capa | Opcional: centavos (gratis con modelo local) |
| 10 | Alucinaciones: causas, mitigación y regresión | Opcional: centavos (gratis con modelo local) |
| 11 | El juez LLM: calibrarlo (opcional) y marcos de evaluación | Opcional: centavos (gratis con modelo local) |
| 12 | **Producción**: caché, actualización incremental, trazas | Casi no: 1 respuesta (gratis con modelo local) |
| 13 | Práctica integrada: el **Medidor de calidad RAG** | Opcional |
| 14 | Cierre | No |

> 💰 Las celdas que llaman al LLM llevan **💰 FYI** y una estimación. Con un modelo de proveedor cada pregunta cuesta ≈ US$ 0.0005; con el modelo local de LM Studio, 0. Las métricas de recuperación **nunca** llaman al LLM: son gratis.

> 📝 Cada parte alterna explicación y código. Lee la explicación **antes** de ejecutar la celda y compara lo que ves con lo que se esperaba.

---
# Parte 0 · Preparar el entorno

In [ ]:
import sys

print("Python:", sys.version.split()[0])
assert sys.version_info[:2] == (3, 12), (
    "Esta sesión está fijada a Python 3.12. Ejecuta `uv sync` dentro de la carpeta "
    "s07-rag-avanzado y selecciona el kernel .venv (ver README)."
)

In [ ]:
import json
import os
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

from almacenes import Fragmento, crear_almacen
from avanzado import (
    RecuperadorDescompuesto, RecuperadorHyDE, RecuperadorSmallToBig, contextualizar, descomponer, fusionar_rrf,
    ingestar_con_metadatos, ingestar_pequenos, texto_con_contexto,
)
from cliente_llm import Gasto, hay_api_key, local_disponible, modelo_local, variable_api_key
from embeddings import embeddings_disponibles, modelo_embeddings, proveedor_embeddings, vector_consulta, vectorizar
from evaluacion import (
    CAPA_DEL_DIAGNOSTICO, SLO_POR_DEFECTO, calibrar_juez, cargar_golden, con_respuesta, contexto_desde_secciones,
    contiene_claves, evaluar_recuperacion_en_k, evaluar_rag, guardar_informe, imprimir_resumen, informe_markdown,
    juzgar_fidelidad, metricas_recuperacion, modelo_juez, numeros_sin_respaldo, proveedor_juez, puntajes_mejor_fragmento,
    tabla_comparativa, tabla_por_tipo, verificar_slo,
)
from grafo import (
    RecuperadorCombinado, RecuperadorGrafo, afectados_por, administrador_de, cargar_grafo, comparar_tripletas,
    dependencias_de, dibujar_grafo, entidades_en, equipos_de, evaluar_multisalto, extraer_tripletas,
    grafo_desde_tripletas, mas_dependidos, resumen_multisalto, vecindad,
)
from rag import (
    NO_SE, citas_en, construir_prompt, fragmentar, generar, ingestar, leer_documento, proveedor_generacion,
    proveedores_disponibles, recuperar, texto_a_vectorizar,
)
from plantillas import renderizar
from embeddings_avanzados import (
    cuantizar_binario, cuantizar_int8, descuantizar_int8, evaluar_matriz, maxsim, modelos_de_embeddings_locales,
    puntajes, puntajes_hamming, truncar, vectores_con,
)
from multimedia import (
    a_fragmentos, cita_de_tiempo, describir_fotograma, evaluar_por_tiempo, extraer_audio, extraer_fotogramas,
    fotograma_mas_cercano, hay_faster_whisper, hay_ffmpeg, medios_disponibles, mmss, transcribir, ventanas_de_tiempo,
)
from produccion import CacheSemantica, IndiceIncremental, Traza

HAY_EMB, detalle = embeddings_disponibles()
print(("✅ Embeddings: " if HAY_EMB else "❌ Embeddings: ") + detalle)

### 🔌 Quién genera las respuestas

El **LLM** que redacta las respuestas se elige con `RAG_PROVIDER` (o `LLM_PROVIDER`) en tu `.env`: `anthropic`, `openai` o `local` (LM Studio, gratis). En esta sesión **el modelo local alcanza para todo**: lo importante es el método de medir, no la nota que saque tu modelo. Con un modelo de proveedor los números mejoran; compáralos.

> El **juez** (Parte 11) se elige con `JUEZ_PROVIDER` y `JUEZ_MODEL` (opcionales). Por defecto juzga el mismo proveedor que genera.

In [ ]:
GEN = proveedor_generacion()
if GEN == "local":
    HAY_LLM, msg = local_disponible(modelo_local())
else:
    HAY_LLM, msg = hay_api_key(GEN), f"falta {variable_api_key(GEN)} en tu .env"
print(f"Generación con: {GEN} · {'✅ disponible' if HAY_LLM else '❌ ' + str(msg)}")
print(f"Juez: {proveedor_juez()} · modelo {modelo_juez()}")

### 🔌 Interruptor general de gasto

💰 **FYI:** con `False`, ninguna celda llama al LLM (las de recuperación siguen funcionando, son gratis). Con `True` y un proveedor disponible, se ejecutan. Si usas un proveedor de pago, configura antes un límite de gasto mensual (ver `docs/COSTOS.md`).

In [ ]:
EJECUTAR_LLAMADAS_DE_PAGO = True   # ← ponla en False para no llamar al LLM
PUEDE_GASTAR = EJECUTAR_LLAMADAS_DE_PAGO and HAY_LLM and HAY_EMB
PUEDE_INDEXAR = HAY_EMB
print("Celdas con LLM activas:", PUEDE_GASTAR)

MANUAL = "datos/manual_soporte.md"
GOLDEN = cargar_golden("datos/golden_set.json")
INFORMES = {}          # nombre → Informe, para comparar al final
gasto = Gasto()        # llamadas sueltas; las evaluaciones guardan su propio costo en cada Informe

def correr(nombre, almacen, **opciones):
    # Evalúa el RAG con el golden set completo y guarda el informe con un nombre.
    inf = evaluar_rag(almacen, GOLDEN, **opciones)
    INFORMES[nombre] = inf
    return inf

---
# Parte 1 · Medir antes de mejorar

## Una analogía de infraestructura

Nadie sube un cambio a producción sin dashboards. En un RAG pasa igual:

| Mundo de infraestructura | Mundo del RAG |
| --- | --- |
| **SLI** (indicador): qué se mide, p. ej. latencia p95 | **Métrica**: Recall@K, datos correctos, alucinaciones… |
| **SLO** (objetivo): el mínimo aceptable, p. ej. 99.9 % | **Umbral de aprobación**: "Recall ≥ 90 %, alucinaciones ≤ 10 %" |
| **Prueba de regresión** al desplegar | **Volver a correr el golden set** al cambiar el prompt, el chunking o el modelo |
| Observabilidad: ¿qué capa falló? | **Diagnóstico por capa**: recuperación vs generación |

Sin medir, cada "mejora" es una opinión. Con un conjunto de preguntas fijo, es un experimento.

## El *golden set* (conjunto de oro)

Es una lista de preguntas con la respuesta correcta **escrita por una persona**, más qué parte del documento la contiene. Es la vara con la que se mide todo lo demás. El de hoy está en `datos/golden_set.json`, tiene **24 preguntas de 6 tipos** y usa el manual de Andes Fibra de la Sesión 5.

| Tipo | Qué es | Qué esperamos del RAG |
| --- | --- | --- |
| `directa` | La respuesta está en un solo lugar | Responder con cita |
| `multi_seccion` | Hacen falta datos de **dos secciones** | Traer ambas y combinarlas |
| `multi_salto` | Hay que **encadenar** un dato con otro (y a veces calcular) | Traer ambas y razonar |
| `ambigua` | Puede entenderse de dos maneras | Cubrir las interpretaciones |
| `fuera_parecida` | Suena a algo del manual, pero **no está** | Decir "no sé" (la trampa) |
| `fuera_ajena` | Nada que ver con el manual | Decir "no sé" |

In [ ]:
from collections import Counter

print(f"{len(GOLDEN)} preguntas · {sum(1 for p in GOLDEN if p['rapido'])} marcadas como 'rápidas' (subconjunto de 10 para evaluar en segundos)\n")
for tipo, n in Counter(p["tipo"] for p in GOLDEN).items():
    print(f"  {tipo:<16}{n:>3}")

print("\nAnatomía de una pregunta:")
ejemplo = next(p for p in GOLDEN if p["id"] == "MS02")
for campo in ("id", "tipo", "pregunta", "respuesta_esperada", "secciones", "claves"):
    print(f"  {campo:<19}{ejemplo[campo]}")

Fíjate en `secciones` y `claves`: la primera dice **dónde** está la respuesta (para medir la **recuperación**) y la segunda **qué dato** debe aparecer (para medir la **generación**). Dos preguntas distintas, dos medidas distintas.

## Las tres métricas de recuperación

Cada fragmento recuperado se marca como **relevante** si pertenece a una de las secciones correctas. Con eso:

| Métrica | Pregunta que responde | Cómo se calcula |
| --- | --- | --- |
| **Recall@K** | ¿Traje **todo** lo que necesitaba? | secciones relevantes que aparecen en el top-K ÷ secciones relevantes que existen |
| **Precision@K** | ¿Cuánto de lo que traje **sirve**? | fragmentos relevantes del top-K ÷ K |
| **MRR** | ¿Qué tan **arriba** quedó el primero útil? | 1 ÷ posición del primer fragmento relevante (0 si no aparece) |

Recall sube al traer más fragmentos; Precision baja (traes más ruido). Casi toda decisión de recuperación es un balance entre las dos: el ruido cuesta tokens, dinero y a veces confunde al modelo.

> 🧮 Con **una sola** sección relevante, Recall@K es lo mismo que *hit@K* de la Sesión 5. Con dos o más (las multi-sección), distingue "traje una" de "traje las dos".

### Un ejemplo calculado a mano

In [ ]:
base = crear_almacen("memoria")
ingestar(base, MANUAL, estrategia="oraciones", max_palabras=70, solape=1)     # el RAG de la Sesión 5
print(f"Base: {base.contar()} fragmentos de ~70 palabras\n")

p = next(x for x in GOLDEN if x["id"] == "M01")
print("Pregunta:", p["pregunta"])
print("Secciones relevantes:", p["secciones"], "\n")
top = recuperar(base, p["pregunta"], k=4)
for i, r in enumerate(top, start=1):
    marca = "✅ relevante" if r.metadatos["seccion"] in p["secciones"] else "·  ruido"
    print(f"  {i}. {r.puntaje:.3f}  {marca:<13} {r.metadatos['seccion']}")

m = metricas_recuperacion(top, p)
print(f"\nRecall@4 = {m['recall']:.2f}  (cubre {int(m['recall'] * len(p['secciones']))} de {len(p['secciones'])} secciones)")
print(f"Precision@4 = {m['precision']:.2f}  ({round(m['precision'] * 4)} de 4 fragmentos sirven)")
print(f"MRR = {m['mrr']:.2f}  (el primero útil quedó en la posición {m['primer_relevante']})")

**Por qué etiquetamos por sección y no por fragmento:** si cambias el tamaño de los fragmentos (lo haremos varias veces hoy), los identificadores cambian pero las secciones no. La misma etiqueta sirve para comparar configuraciones distintas.

**Una limitación honesta:** con 24 preguntas, una diferencia de 1 pregunta (≈ 4 puntos) es ruido. El golden set sirve para **comparar ideas**, no para prometer exactitud; uno real necesita decenas o cientos de preguntas reales de usuarios.

---
# Parte 2 · Dónde falla el RAG básico

Medimos el RAG de la Sesión 5 (fragmentos de ~70 palabras, k = 4) **solo en recuperación** (gratis) y lo desglosamos **por tipo de pregunta**. Los promedios globales esconden justo lo interesante.

In [ ]:
inf_base_rec = evaluar_rag(base, GOLDEN, k=4, con_generacion=False)
print(tabla_por_tipo(inf_base_rec))
res = inf_base_rec.resumen()
print(f"\nTodas · Recall@4 = {res['recall']:.0%} · Precision@4 = {res['precision']:.0%} · MRR = {res['mrr']:.0%} · "
      f"{res['palabras_contexto']:.0f} palabras de contexto por pregunta")

In [ ]:
print("Preguntas donde NO llegó toda la información necesaria:\n")
for f in inf_base_rec.filas:
    if f.recall is not None and f.recall < 1:
        p = next(x for x in GOLDEN if x["id"] == f.id)
        print(f"{f.id} ({f.tipo}) recall={f.recall:.2f}")
        print(f"   pregunta : {f.pregunta}")
        print(f"   esperadas: {p['secciones']}")
        print(f"   llegaron : {sorted(set(f.secciones_recuperadas))}\n")

### Lo que se ve

- Las preguntas **directas** casi siempre salen bien: un buen RAG básico hace bien lo básico.
- Las que necesitan **dos temas** (multi-sección y, sobre todo, multi-salto) son las que pierden información: el vector de una pregunta larga se parece **un poco a cada tema y mucho a ninguno**, y los fragmentos del segundo tema quedan fuera del top-4.
- **Subir `k` ayuda, pero no es gratis.** Veámoslo con la curva clásica.

In [ ]:
curva = evaluar_recuperacion_en_k(base, GOLDEN, ks=(1, 2, 3, 4, 6, 8))
ks = list(curva)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(ks, [curva[k]["recall"] for k in ks], "o-", label="Recall@K")
ax.plot(ks, [curva[k]["precision"] for k in ks], "s-", label="Precision@K")
ax.set_xlabel("K (fragmentos entregados al modelo)"); ax.set_ylabel("proporción"); ax.set_ylim(0, 1.05)
ax.set_title("Más K: más cobertura, más ruido"); ax.legend(); ax.grid(alpha=0.3)
plt.show()
for k in ks:
    print(f"K={k}: Recall {curva[k]['recall']:.0%} · Precision {curva[k]['precision']:.0%} · MRR {curva[k]['mrr']:.0%}")

Las dos curvas se cruzan en sentido contrario: **la forma de subir Recall sin hundir Precision es ser más listo, no traer más**. De eso tratan las técnicas que siguen.

> 💬 **Para pensar:** si Recall@8 llega casi a 100 %, ¿por qué no usar siempre K = 8? Piensa en tokens, costo, latencia y en lo que hace un modelo pequeño cuando recibe ocho fragmentos y solo dos sirven.

---
# Parte 3 · *Small-to-big*, encabezados y *contextual retrieval*

## El dilema del tamaño del fragmento

| Fragmento | Ventaja | Problema |
| --- | --- | --- |
| **Pequeño** (≈ 35 palabras) | Su vector habla de **una sola cosa**: se recupera con precisión | Al modelo le falta contexto para responder |
| **Grande** (una sección) | Tiene contexto completo | Su vector se **diluye**; entra ruido al prompt |

## La idea: no elegir

```
 BUSCAR (precisión)                         ENTREGAR (contexto)
 fragmento pequeño  ───►  su "padre":  la sección completa     (modo "seccion")
                                       o él + sus vecinos      (modo "ventana")
```

Se **indexan** los fragmentos pequeños, pero al modelo se le **entrega** el trozo grande al que pertenecen (el *parent document*, "documento padre"). Es el truco detrás de muchos retrievers "parent-document" y "sentence-window".

## Encabezados contextuales

Un fragmento suelto ("Cuesta 60 dólares.") es ambiguo. Anteponerle su **encabezado** (la sección, el título del documento) antes de vectorizarlo le da el tema. Es barato y no cambia lo que se entrega. En nuestro `rag.py` ya se antepone la sección (`texto_a_vectorizar`); aquí **medimos** si importa, quitándola.

> Ambas técnicas son gratis: solo usan embeddings locales.

In [ ]:
# Fragmentos pequeños (35 palabras), con y sin encabezado contextual
peq = crear_almacen("memoria"); frs_peq = ingestar_pequenos(peq, MANUAL, max_palabras=35, encabezado=True)
peq_sin = crear_almacen("memoria"); ingestar_pequenos(peq_sin, MANUAL, max_palabras=35, encabezado=False)
print(f"{len(frs_peq)} fragmentos pequeños (frente a {base.contar()} de ~70 palabras)\n")

informes = {
    "70 pal. k=4": inf_base_rec,
    "35 pal. k=4": evaluar_rag(peq, GOLDEN, k=4, con_generacion=False),
    "35 pal. sin enc.": evaluar_rag(peq_sin, GOLDEN, k=4, con_generacion=False),
}
print(tabla_comparativa(informes, ("recall", "precision", "mrr")))

Fíjate en dos cosas: los fragmentos pequeños mejoran **Precision** (cada fragmento habla de lo suyo), y quitar el encabezado casi no cambia el resultado **en este manual**, porque las secciones ya son temáticas y el texto de cada fragmento nombra el tema. En documentos donde los fragmentos son genéricos ("Ver tabla anterior"), el encabezado pesa mucho más. **Medir te dice cuándo una técnica importa y cuándo es adorno.**

### Ahora, *small-to-big*: buscar pequeño, entregar grande

In [ ]:
informes_s2b = {"base 70 pal. k=4": inf_base_rec}
for modo in ("seccion", "ventana"):
    rec = RecuperadorSmallToBig(peq, MANUAL, frs_peq, modo=modo)
    for k in (2, 3):
        informes_s2b[f"{modo} k={k}"] = evaluar_rag(None, GOLDEN, k=k, con_generacion=False, recuperador=rec)
print(tabla_comparativa(informes_s2b, ("recall", "precision", "mrr")))

fig, ax = plt.subplots(figsize=(6.5, 3.8))
for nombre, inf in informes_s2b.items():
    r = inf.resumen()
    ax.scatter(r["palabras_contexto"], r["recall"], s=70)
    ax.annotate(nombre, (r["palabras_contexto"], r["recall"]), fontsize=7, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("palabras de contexto por pregunta (≈ costo)"); ax.set_ylabel("Recall"); ax.grid(alpha=0.3)
ax.set_title("Cobertura vs tamaño del contexto"); plt.show()

### Cómo leer el resultado

- **`seccion`** entrega secciones completas: el Recall sube (el modelo recibe el contexto entero) pero **la Precision baja** y el contexto crece. Es la opción cuando las respuestas dependen de condiciones que están en el resto de la sección.
- **`ventana`** entrega el fragmento y sus vecinos: contexto compacto y buena Precision; conviene cuando el vecino inmediato basta.
- **No hay ganador universal**: depende de cuánto contexto necesite tu respuesta y de cuánto estés dispuesto a pagar en tokens. Por eso se mide con **tus** preguntas.

> ⚠️ Small-to-big **no arregla** lo que está en dos secciones distintas si ninguna de las dos aparece en el top-K de hijos. Para eso, la descomposición (Parte 4), después de una variante de los encabezados que usa un LLM.

### Contextual Retrieval: que un LLM escriba el contexto

**🧠 Qué es.** Un encabezado ("3. Facturación y pagos") da el tema, pero un fragmento puede seguir sin decir **de qué documento ni de qué parte** habla. En el *contextual retrieval*, un LLM lee el documento completo y cada fragmento, y escribe **una frase de contexto**. Esa frase se antepone al fragmento **antes de vectorizarlo** (y antes de construir un índice BM25, si lo hay). Es la versión "automática" del encabezado contextual.

**⚙️ Qué hace.** Cambia el texto que se **vectoriza**, no el que se **entrega** al modelo. Cuesta **una llamada al LLM por fragmento, una sola vez al indexar** (no por pregunta). Como el documento completo viaja en *cada* llamada, es el caso ideal para la **caché de prompt** de los proveedores (Parte 12).

**🎯 Para qué sirve.** Para fragmentos que sueltos son ambiguos ("Cuesta 60 dólares.", "Ver la tabla anterior.") en corpus largos con muchos documentos parecidos.

**🏗️ Analogía de infraestructura.** Es añadir **etiquetas y una descripción** a un recurso (tags en la nube, el campo *description* de la CMDB) para que el buscador lo encuentre aunque su nombre no sea claro.

**✏️ Ejemplo a mano (ilustrativo).**

| Fragmento | Frase de contexto que escribe el LLM | Texto que se vectoriza |
| --- | --- | --- |
| "Cuesta 60 dólares." | "Costo de la instalación cuando el cliente no firma permanencia." | contexto + fragmento |

**⚠️ Cuándo no.** Si tus fragmentos ya nombran su tema o el encabezado basta (como en este manual, ya lo medimos arriba). Cuesta tiempo y dinero al indexar; el LLM puede **inventar** (un contexto equivocado empeora la búsqueda); y hay que **regenerarlo** cuando cambia el documento.

**🐞 Errores típicos.** (1) Activarlo sin medir. (2) Contextos que repiten el fragmento o inventan datos. (3) No regenerar el contexto al editar el documento. (4) Olvidar que el costo crece con *tamaño del documento × número de fragmentos*.

Primero, los dos preparativos gratis (los mismos 26 fragmentos de la base y una función que arma un almacén con el texto que quieras vectorizar):

In [ ]:
documento_manual = Path(MANUAL).read_text(encoding="utf-8")
frs_base = fragmentar(leer_documento(MANUAL), "oraciones", 70, 1)        # los mismos 26 fragmentos de la base

def almacen_de(textos):
    # Un almacén con los MISMOS fragmentos, pero vectorizando el texto que se indique.
    a = crear_almacen("memoria")
    vectores = vectorizar(textos, tipo="documento")
    a.preparar(len(vectores[0]))
    a.insertar([Fragmento(f.id, f.texto, f.metadatos, v) for f, v in zip(frs_base, vectores)])
    return a

print(f"{len(frs_base)} fragmentos · documento de {len(documento_manual.split())} palabras (viajará completo en cada llamada)")

In [ ]:
# 💰 FYI: esta celda consume dinero. 26 llamadas, cada una con el manual completo (≈ 58 000 tokens de entrada en total con el modelo local, ≈ 86 000 medidos con el modelo económico de Anthropic); gratis con el local
if PUEDE_GASTAR:
    t0 = time.perf_counter()
    contextos, llamadas = contextualizar(frs_base, documento_manual)
    for l in llamadas: gasto.registrar(l)
    print(f"{len(frs_base)} fragmentos contextualizados en {time.perf_counter() - t0:.0f} s · "
          f"{sum(l.tokens_entrada for l in llamadas):,} tokens de entrada · {sum(l.tokens_salida for l in llamadas):,} de salida\n")
    for f in frs_base[1:4]:
        print(f"[{f.metadatos['seccion']}] {f.texto[:75]}…\n      contexto → {contextos[f.id]}\n")

    variantes = {
        "sin encabezado ni contexto": [f.texto for f in frs_base],
        "con encabezado (la base)": [texto_a_vectorizar(f) for f in frs_base],
        "solo contexto del LLM": [texto_con_contexto(f, contextos[f.id]) for f in frs_base],
        "encabezado + contexto": [texto_con_contexto(f, contextos[f.id], True) for f in frs_base],
    }
    inf_ctx = {nombre: evaluar_rag(almacen_de(textos), GOLDEN, k=4, con_generacion=False) for nombre, textos in variantes.items()}
    print(tabla_comparativa(inf_ctx, ("recall", "precision", "mrr")))
else:
    print("⚪ Celda de pago desactivada.")

### Cómo leer este resultado

- Compara con la fila **"con encabezado"**, que es lo que ya hacíamos gratis: el contexto del LLM tiene que ganarle **por más que el ruido** (1 pregunta ≈ 4 puntos) para justificar sus llamadas.
- En este manual el efecto es **pequeño**, igual que el del encabezado: las secciones ya son temáticas y cada fragmento nombra su tema. Es el mismo mensaje de siempre: **medir te dice cuándo una técnica importa y cuándo es adorno.** Donde sí pesa es en corpus grandes, con fragmentos genéricos o documentos que se parecen entre sí.
- **Una referencia medida:** con el modelo local de 3B (esta corrida) el contexto del LLM **no superó al encabezado gratis**: recall igual (92 %), precisión de 61 % a 66 % y MRR de 0.89 a 0.92 según la variante. Probado una vez con el modelo económico de un proveedor, el recall subió a 95 %-97 % y el MRR quedó en 0.90-0.91, también sin superar al encabezado (0.92). Con 20 preguntas, esas diferencias son **ruido**.
- Mira también las frases generadas: con un modelo pequeño algunas **repiten el encabezado** o se pasan de largas. Un contexto malo no ayuda, y nadie lo revisa si no lo imprimes.

---
# Parte 4 · Descomposición de preguntas y HyDE

## Preguntas multi-salto

> *"Me suspendieron por falta de pago y voy a pagar hoy en una agencia, ¿en cuánto tiempo se reactiva el servicio y qué documento debo llevar?"*

Para responder hay que unir **dos datos de dos secciones**: cuánto tarda la reactivación (facturación y pagos) y qué documento piden en la agencia (contacto de agencias). Una sola búsqueda con la pregunta completa produce **un** vector que no se parece del todo a ninguna de las dos.

## Descomposición

Pedimos a un LLM que **separe la pregunta en subpreguntas** de un solo tema, buscamos **cada una** y fusionamos los resultados con **RRF** (*reciprocal rank fusion*: cada fragmento suma 1 ÷ (60 + posición) en cada lista donde aparece, así que no hace falta comparar puntajes entre búsquedas distintas).

```
pregunta ──► LLM ──► subpregunta 1 ──► buscar ─┐
                     subpregunta 2 ──► buscar ─┼──► fusión RRF ──► top-k ──► prompt
             pregunta original ───────► buscar ─┘
```

💰 **FYI:** esta técnica añade **1 llamada corta** al LLM por pregunta (≈ US$ 0.0002 con un modelo de proveedor; gratis con el local).

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.0006 con un modelo de proveedor; gratis con el modelo local)
if PUEDE_GASTAR:
    for id_ in ("MS01", "MS03", "G01"):
        p = next(x for x in GOLDEN if x["id"] == id_)
        subs, llamadas = descomponer(p["pregunta"])
        for l in llamadas: gasto.registrar(l)
        print(f"{id_}: {p['pregunta']}")
        for s in subs: print(f"     → {s}")
        print()
else:
    print("⚪ Celda de pago desactivada.")

Observa que una pregunta de **un solo tema** (`G01`) se descompone en lo mismo o en variantes: la técnica no estorba, pero cuesta una llamada. Ahora la medimos (solo recuperación) y la comparamos con la base.

> 📖 **HyDE es de lectura.** En clase se resume en una línea dentro de la descomposición; la celda que sigue lo mide por si quieres ver el número, pero puedes omitirla.

## HyDE (*Hypothetical Document Embeddings*)

Una **pregunta** y su **respuesta** se parecen menos de lo que crees: "¿Hasta cuándo puedo pagar?" y "La factura vence el día 15…" casi no comparten palabras. HyDE pide al LLM que **escriba una respuesta hipotética** (aunque invente los datos) y **busca con el vector de ese texto**, que se parece más a los fragmentos del manual. El texto inventado jamás se muestra al usuario: solo sirve para buscar.

Es una de varias formas de *transformar la consulta* antes de buscar (otra: generar varias paráfrasis y fusionar, "multi-consulta"). La **reescritura de consulta** como tal la viste en la Sesión 6.

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.01 con un modelo de proveedor: 2 técnicas × 24 preguntas × 1 llamada corta; gratis con el modelo local)
if PUEDE_GASTAR:
    t0 = time.perf_counter()
    desc = RecuperadorDescompuesto(base, k_sub=3)
    hyde = RecuperadorHyDE(base)
    informes_q = {
        "base k=4": inf_base_rec,
        "descomposición": evaluar_rag(None, GOLDEN, k=4, con_generacion=False, recuperador=desc),
        "HyDE": evaluar_rag(None, GOLDEN, k=4, con_generacion=False, recuperador=hyde),
    }
    print(tabla_comparativa(informes_q, ("recall", "precision", "mrr", "hit")))
    print(f"\n({time.perf_counter() - t0:.0f} s)\n")
    for nombre, inf in informes_q.items():
        pt = inf.por_tipo()
        print(f"{nombre:<16}" + "  ".join(f"{t}: {pt[t]['recall']:.0%}" for t in pt if pt[t]["n_con_respuesta"]))
else:
    print("⚪ Celda de pago desactivada.")

### Cómo leerlo

- La **descomposición** debería ayudar justo en `multi_salto` y `multi_seccion`, que son las preguntas para las que existe. Si en `directa` no cambia nada, está bien: no era su trabajo.
- **HyDE** es más **irregular**: cuando el modelo inventa una respuesta con el vocabulario correcto, ayuda; cuando inventa algo ajeno, te lleva a un fragmento equivocado. Con un modelo pequeño, suele empeorar la precisión y el MRR. Es una técnica para **probar y medir**, no para activar por defecto.
- Fíjate también en el costo en segundos: cada técnica con LLM añade latencia a **cada** pregunta.

> 🧰 **Regla práctica:** usa una técnica cuando puedas nombrar **la falla que ataca** y puedas **medir** que la arregla. Si no, es complejidad sin evidencia.

---
# Parte 5 · GraphRAG: qué es y cuándo conviene

## Un problema que los vectores no resuelven bien

Cambiamos de documento. Ahora es el **catálogo de servicios de TI** de Andes Fibra (ficticio), el tipo de cosa que vive en una **CMDB** (base de configuración de una mesa de ayuda): cada servicio, de qué depende y quién lo administra.

> *"Si falla la base de datos de clientes, ¿qué servicios dejan de funcionar?"*

El catálogo solo dice, en la sección de cada servicio, **de qué depende**. **Nadie escribió** "de la base de datos dependen X, Y y Z". La respuesta hay que **construirla recorriendo las dependencias al revés**, varios saltos. Un RAG vectorial busca *texto parecido a la pregunta*: encuentra la sección de la base de datos, que no contiene la respuesta.

## ¿Qué es GraphRAG?

**Un RAG que, además de fragmentos de texto, usa un grafo de conocimiento.**

- **Grafo de conocimiento:** una red de **entidades** (nodos: servicios, equipos, personas, productos) unidas por **relaciones** (aristas: "depende de", "administrado por"). Cada hecho es una **tripleta** `(sujeto, relación, objeto)`.
- **Pipeline típico:**

```
 documentos ──► (LLM) extrae entidades y relaciones ──► grafo de conocimiento
                                                              │
        ┌───────────── búsqueda LOCAL ─────────────┐          │
        │  entidades de la pregunta → sus vecinos  │◄─────────┤
        └──────────────────────────────────────────┘          │
        ┌───────────── búsqueda GLOBAL ────────────┐          │
        │  comunidades del grafo → resúmenes       │◄─────────┘
        └──────────────────────────────────────────┘
                              │
                     contexto ──► LLM ──► respuesta
```

## Implementaciones reales

| Proyecto | Qué es (según su documentación) |
| --- | --- |
| **[Microsoft GraphRAG](https://microsoft.github.io/graphrag/)** | Enfoque "estructurado y jerárquico" del RAG: divide el corpus en *text units*, **extrae entidades, relaciones y afirmaciones con un LLM**, agrupa el grafo en **comunidades** (algoritmo de Leiden) y **genera un resumen por comunidad**. Modos de consulta: **global** (preguntas sobre todo el corpus, con los resúmenes), **local** (entidades y sus vecinos), **DRIFT** (local con contexto de comunidad) y **básica** (búsqueda vectorial clásica). Su propia documentación recomienda *ajustar los prompts* antes de fiarse de los resultados. |
| **[LightRAG](https://github.com/HKUDS/LightRAG)** | Alternativa "ligera" (paper en EMNLP 2025, MIT): guarda **grafo y vectores a la vez**, un LLM extrae entidades y relaciones, y ofrece modos *naive*, *local*, *global*, *hybrid* y *mix* (el predeterminado). Permite **añadir documentos sin reindexar todo**, y evita los resúmenes por comunidad para gastar menos llamadas al LLM. |

> 🚫 **No los instalamos hoy.** Son paquetes pesados y su indexado hace **muchas** llamadas al LLM (cuesta dinero y tiempo). Hoy construimos a escala de juguete, con `networkx`, **lo que hay dentro**: tripletas, grafo, recorrido y contexto. Con eso entiendes qué hacen y qué te cuestan.

## ¿Cuándo aporta y cuándo NO?

| Aporta cuando… | No compensa cuando… |
| --- | --- |
| Las preguntas son **relacionales** ("¿qué se afecta si cae X?", "¿quién depende de quién?") | Las preguntas son **directas** y la respuesta está en un párrafo |
| Son **globales** ("¿cuál es el tema principal del corpus?", "¿cuál es el servicio más crítico?") | Tu corpus es pequeño y cabe en el contexto |
| Hay entidades con **muchas conexiones** (CMDB, contratos, redes de personas, cadena de suministro) | Tus documentos cambian a diario (**reindexar el grafo cuesta**) |
| Necesitas **explicar el camino** de la respuesta | No puedes validar la **calidad de la extracción** (un grafo con errores contamina todo) |

**Costos que casi nadie cuenta:** extraer el grafo = **1 o más llamadas al LLM por fragmento** del corpus (en un corpus grande, miles); mantenerlo al día; y revisar su calidad. Antes de adoptar GraphRAG, pregúntate si un RAG con buen *chunking*, *small-to-big* y descomposición ya resuelve tus preguntas.

In [ ]:
catalogo = leer_documento("datos/servicios_ti.md")
print(f"{len(catalogo.secciones)} secciones · ejemplo:\n")
for titulo, texto in catalogo.secciones[10:13]:
    print(f"## {titulo}\n{texto}\n")

Observa el patrón: cada sección dice **de qué depende** ese servicio, nunca **qué depende de él**. Esa información está *implícita*, repartida por todas las demás secciones.

### El grafo curado a mano

Para tener una base fiable, las tripletas del catálogo están escritas a mano en `datos/servicios_ti_grafo.json` (40 tripletas, 24 entidades). **Es el camino fiable**: si el grafo está mal, todo lo demás está mal. Más abajo veremos qué pasa cuando lo extrae un LLM.

In [ ]:
G = cargar_grafo()
print(f"{G.number_of_nodes()} entidades · {G.number_of_edges()} relaciones\n")
for a, b, d in list(G.edges(data=True))[:6]:
    print(f"  ({a}) ──{d['relacion']}──► ({b})")

dibujar_grafo(G, titulo="Servicios de TI de Andes Fibra · A → B significa 'A depende de B'")
plt.show()

### El grafo se consulta como una base de datos

Con el grafo, la pregunta de impacto **no necesita un LLM**: es un algoritmo (recorrer las aristas al revés).

In [ ]:
nodo = "Base de datos de clientes"
afectados = afectados_por(G, nodo)
print(f"Si falla «{nodo}», dejan de funcionar {len(afectados)} servicios:")
for s in afectados: print("   •", s)

print("\n¿De qué depende el Chat de soporte, directa o indirectamente?")
print("   ", ", ".join(dependencias_de(G, "Chat de soporte")))

print("\nServicios con más servicios que dependen DIRECTAMENTE de ellos (pregunta 'global': hay que contar):")
for n, c in mas_dependidos(G): print(f"   {c} · {n}")

dibujar_grafo(G, resaltar=[nodo, *afectados], titulo=f"Si falla «{nodo}» (rojo = origen y afectados)")
plt.show()

Eso es lo que **un grafo da y un vector no**: *recorrer*, *contar*, *agregar*. Ahora veamos qué hace la búsqueda vectorial con la misma pregunta.

In [ ]:
vec = crear_almacen("memoria")
ingestar(vec, "datos/servicios_ti.md", estrategia="oraciones", max_palabras=70, solape=1)

pregunta = "Si falla la base de datos de clientes, ¿qué servicios dejan de funcionar?"
print(pregunta, "\n\nTop-4 vectorial:\n")
for i, r in enumerate(recuperar(vec, pregunta, k=4), start=1):
    print(f"  {i}. {r.puntaje:.3f}  {r.metadatos['seccion']}")
print("\nDebería traer (porque dependen de ella, directa o indirectamente):", ", ".join(afectados))

El vectorial trae los fragmentos **sobre** la base de datos y algunos servicios que la *nombran*, pero deja fuera varios afectados **indirectos** (el chat de soporte depende de tickets, que depende de la base de datos: dos saltos). Es exactamente el tipo de pregunta para el que se inventó GraphRAG.

---
# Parte 6 · GraphRAG medido: vectorial vs grafo vs combinado

Hay 8 preguntas **multi-salto** en `datos/preguntas_multisalto.json`, con la lista exacta de entidades que debe contener la respuesta (calculada con el grafo curado). Medimos **dos cosas por separado**, como siempre:

| Capa | Métrica | Pregunta |
| --- | --- | --- |
| Recuperación | **cobertura del contexto** | ¿Las entidades de la respuesta llegaron al prompt? |
| Generación | **recall y precisión de la respuesta** | ¿El modelo nombró todas y solo esas? |

## Los recuperadores que comparamos

| Recuperador | Qué entrega al modelo |
| --- | --- |
| `vectorial` (k = 4 y k = 8) | Los fragmentos del catálogo más parecidos a la pregunta |
| `grafo · vecindad` | Los **hechos** a pocos saltos de las entidades de la pregunta (*búsqueda local*) |
| `grafo · recorrido` | Esos hechos **más un análisis calculado** con un algoritmo del grafo (impacto, dependencias, ranking) |
| `vectorial + recorrido` | Lo de los dos mundos juntos |

Primero, **solo recuperación** (gratis): ¿llega la información al prompt?

In [ ]:
PREGUNTAS_MH = json.loads(Path("datos/preguntas_multisalto.json").read_text(encoding="utf-8"))
rv = lambda pregunta, k: recuperar(vec, pregunta, k=k)

RECUPERADORES = {
    "vectorial k=4": (rv, 4),
    "vectorial k=8": (rv, 8),
    "grafo vecindad (1 salto)": (RecuperadorGrafo(G, profundidad=1), 4),
    "grafo vecindad (2 saltos)": (RecuperadorGrafo(G, profundidad=2), 4),
    "grafo vecindad (3 saltos)": (RecuperadorGrafo(G, profundidad=3), 4),
    "grafo recorrido": (RecuperadorGrafo(G, profundidad=2, modo="recorrido"), 4),
    "vectorial + recorrido": (RecuperadorCombinado(rv, RecuperadorGrafo(G, profundidad=2, modo="recorrido")), 4),
}
print(f"{'recuperador':<28}{'cobertura':>10}{'palabras':>10}   por pregunta (MH1..MH8)")
for nombre, (rec, k) in RECUPERADORES.items():
    filas = evaluar_multisalto(G, PREGUNTAS_MH, rec, k=k, con_generacion=False)
    s = resumen_multisalto(filas)
    print(f"{nombre:<28}{s['contexto_recall']:>10.0%}{s['palabras_contexto']:>10.0f}   " + " ".join(f"{f['contexto_recall']:.1f}" for f in filas))

### Lo que se ve

- **Vectorial**: subir `k` mejora la cobertura, pero a costa de **más palabras** de contexto, y aun así deja agujeros.
- **Grafo por vecindad**: cada salto de profundidad trae más hechos; con 3 saltos casi todo llega... pero también llega más ruido. La profundidad es un *k* disfrazado.
- **Recorrido**: el algoritmo entrega **exactamente** lo que se pidió, con poco texto.
- **Combinado**: cubre también las preguntas donde la entidad no aparece en el texto de la pregunta (la "global").

Pero *recuperar bien* no es *responder bien*. Ahora la **generación** (💰 una llamada por pregunta, gratis con el modelo local):

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.002 por recuperador con un modelo de proveedor: 8 llamadas cada uno; gratis con el modelo local)
if PUEDE_GASTAR:
    resultados_mh = {}
    for nombre in ("vectorial k=4", "vectorial k=8", "grafo vecindad (3 saltos)", "grafo recorrido", "vectorial + recorrido"):
        rec, k = RECUPERADORES[nombre]
        resultados_mh[nombre] = evaluar_multisalto(G, PREGUNTAS_MH, rec, k=k, version="v2")
    print(f"{'recuperador':<28}{'ctx':>6}{'resp. recall':>14}{'resp. precisión':>17}{'completas':>11}")
    for nombre, filas in resultados_mh.items():
        s = resumen_multisalto(filas)
        print(f"{nombre:<28}{s['contexto_recall']:>6.0%}{s['respuesta_recall']:>14.0%}{s['respuesta_precision']:>17.0%}{s['completas']:>8}/{s['n']}")
else:
    print("⚪ Celda de pago desactivada.")

Compara las columnas **ctx** (¿llegó?) y **resp. recall** (¿lo usó?). Si en una fila la primera es alta y la segunda baja, **el problema ya no es la recuperación**: el modelo tiene los hechos pero no sabe encadenarlos. Un modelo pequeño con una lista de hechos de dos o tres saltos suele fallar; por eso el **recorrido** (que le entrega el resultado ya calculado) rinde mucho más.

Veamos respuestas concretas:

In [ ]:
if PUEDE_GASTAR:
    for id_ in ("MH1", "MH4", "MH7"):
        p = next(x for x in PREGUNTAS_MH if x["id"] == id_)
        print(f"{id_} · {p['pregunta']}")
        print(f"   esperado: {', '.join(p['esperadas'])}")
        for nombre in ("vectorial k=4", "grafo recorrido"):
            fila = next(f for f in resultados_mh[nombre] if f["id"] == id_)
            print(f"   {nombre:<16}: {fila['respuesta'][:170]!r}  (faltan {fila['faltan']})")
        print()

**`MH4`** ("¿qué equipo administra el servicio donde guarda sus datos la API de facturación?") suele fallar con **todos** los recuperadores: el hecho está en el contexto, pero el modelo contesta con el equipo de la *API*, no el de la *base de datos*. Es una falla de **razonamiento encadenado** (generación), no de recuperación. Ningún recuperador la arregla; la arreglan un modelo más fuerte o un prompt que obligue a razonar paso a paso.

---
## El paso frágil: extraer el grafo con un LLM

Hasta aquí usamos un grafo **curado a mano**. En la vida real nadie escribe 40 tripletas por documento: **un LLM las extrae del texto**. Es, de lejos, la parte más cara y más frágil de GraphRAG, y la que casi nunca se mide. Midámosla contra el grafo curado, con dos versiones del prompt:

- **v1**: nombres libres.
- **v2**: se le da la **lista de entidades conocidas** y debe usar solo esas.

💰 **FYI:** 1 llamada por sección (16 secciones por versión); gratis con el modelo local, ≈ US$ 0.003 con un modelo de proveedor.

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.006 con un modelo de proveedor: 2 versiones × 16 secciones; gratis con el modelo local)
if PUEDE_GASTAR:
    extraccion = {}
    for version in ("v1", "v2"):
        t0 = time.perf_counter()
        tripletas, llamadas = extraer_tripletas("datos/servicios_ti.md", version=version, entidades=list(G.nodes))
        for l in llamadas: gasto.registrar(l)
        c = comparar_tripletas(G, tripletas)
        extraccion[version] = (tripletas, c)
        print(f"Prompt {version} · {time.perf_counter() - t0:.0f} s · {c['extraidas']} tripletas extraídas "
              f"(el grafo curado tiene {c['curadas']})")
        print(f"   correctas {c['correctas']} · precisión {c['precision']:.0%} · recall {c['recall']:.0%} · F1 {c['f1']:.0%}")
        print(f"   con una entidad que no existe en el catálogo: {c['con_entidad_no_reconocida']}  {c['nombres_no_reconocidos'][:4]}")
        print(f"   tripletas inventadas o mal formadas (muestra): {c['sobran'][:3]}\n")
else:
    print("⚪ Celda de pago desactivada.")

### Lo que enseña esta medición

1. **Con nombres libres (v1)** el modelo escribe "Directorio de usuarios y Base de datos de clientes" como *una* entidad, o "centros de datos", y esas aristas **no se pueden unir** al resto del grafo.
2. **Darle el vocabulario (v2)** mejora mucho el *recall* (encuentra casi todo), pero **la precisión sigue lejos del 100 %**: inventa relaciones que el texto no dice (aristas de más) o invierte el sentido.
3. **Una arista falsa no es un error aislado**: cambia la respuesta de **todos los recorridos** que pasan por ella. Veámoslo.

In [ ]:
if PUEDE_GASTAR:
    G_ext = grafo_desde_tripletas(G, extraccion["v2"][0])
    print(f"Grafo extraído (v2): {G_ext.number_of_edges()} relaciones (el curado tiene {G.number_of_edges()})\n")
    print(f"{'pregunta':<6}{'curado':>8}{'extraído':>10}   ¿qué cambia?")
    for p in PREGUNTAS_MH:
        if p["tipo"] != "impacto":
            continue
        nodo = entidades_en(G, p["pregunta"])[0]
        bien, mal = set(afectados_por(G, nodo)), set(afectados_por(G_ext, nodo))
        sobran, faltan = sorted(mal - bien), sorted(bien - mal)
        print(f"{p['id']:<6}{len(bien):>8}{len(mal):>10}   sobran {len(sobran)}" + (f" ({', '.join(sobran[:3])}…)" if sobran else "")
              + f" · faltan {len(faltan)}")

**Moraleja:** el grafo es tan bueno como su extracción, y **hay que medir la extracción** (precisión y recall de tripletas contra una muestra curada) **antes** de confiar en las respuestas. Es el mismo principio de hoy aplicado a otra capa: *no se confía en lo que no se mide*.


> ℹ️ El *RAG agéntico* (un modelo que decide qué buscar y cuándo parar) se verá en sesiones posteriores.

---
# Parte 7 · Modelos de recuperación modernos y embeddings

Hasta aquí todos nuestros recuperadores usaron **un vector por fragmento** (y BM25 en la Sesión 6). Hay otras formas de representar y comparar textos. Esta parte las ubica y, sobre todo, **mide** dos formas de hacer los vectores más baratos: **Matryoshka** (vectores más cortos) y **cuantización** (números más pequeños). Todo es gratis: solo embeddings locales y numpy.

## Tres familias de recuperadores

**🧠 Qué es.** Un recuperador convierte pregunta y fragmento en algo comparable. Cambia **qué guarda** por cada texto y **cómo compara**:

| Familia | Qué guarda por texto | Cómo compara | Ejemplos | Su fortaleza | Su costo |
| --- | --- | --- | --- | --- | --- |
| **Léxica** | Las palabras, con más peso a las raras (IDF) | Palabras exactas en común | BM25 (Sesión 6) | Códigos, nombres, cifras exactas; no necesita modelo | Muy barata |
| **Densa, un vector** (*bi-encoder*) | **Un** vector por texto (p. ej. 768 números) | Coseno entre el vector de la pregunta y el del fragmento | `nomic`, `mxbai`, los de OpenAI | Significado, paráfrasis, otros idiomas | Un vector por fragmento |
| **Dispersa aprendida** | Un vector del tamaño del vocabulario (decenas de miles de posiciones), casi todo ceros: un **peso por palabra**, incluidas palabras relacionadas que *no* aparecen (**expansión**) | Producto punto de pesos; se guarda en un índice invertido como BM25 | **SPLADE** | Mezcla lo léxico con lo semántico y se indexa como BM25 | Un modelo para indexar y otro paso para consultar |
| **Interacción tardía** (*late interaction*) | **Un vector por token** | Para cada token de la pregunta, su **mejor coincidencia** en el documento (*MaxSim*); se suman | **ColBERT**; **ColPali / ColQwen** (sobre imágenes de páginas) | Precisión fina; no diluye un documento largo en un solo vector | **Mucho** más almacenamiento (un vector por token) |

**⚙️ Qué hace cada una, en una frase.** BM25 *cuenta palabras*; el bi-encoder *resume un texto en un punto*; SPLADE *reparte pesos entre palabras (y sus parientes)*; la interacción tardía *deja que cada palabra de la pregunta busque su pareja* en el documento. **ColPali** aplica esa última idea a la **imagen de una página** (cada trozo de la imagen es un "token"), así que sirve para PDF con tablas, gráficos y diseño sin pasar por OCR.

**🎯 Para qué sirve conocerlas.** Para saber qué hay cuando el bi-encoder + BM25 + reranking (Sesión 6) se queda corto, y para leer la documentación de una base vectorial que anuncie "vectores dispersos" o "multivector". **Hoy no las implementamos** (SPLADE y ColBERT necesitan sus propios modelos); sí implementamos el cálculo de MaxSim a mano.

**🏗️ Analogía de infraestructura.** BM25 es buscar con `grep` ponderado; el bi-encoder es buscar por la *etiqueta* general del recurso; SPLADE es un `grep` al que alguien añadió **sinónimos** a cada documento; la interacción tardía es comparar **línea por línea** en lugar de comparar el resumen.

**✏️ Ejemplo a mano (SPLADE, ilustrativo).** El fragmento "Cuesta 60 dólares la instalación" se guardaría con pesos como `instalación 2.1 · cuesta 1.4 · dólares 1.2 · precio 0.9 · costo 0.8 · instalar 0.7`: las últimas tres palabras **no aparecen** en el texto; el modelo las añade porque suelen ir con él. Así una pregunta con "precio" lo encuentra por coincidencia de palabras. *(Pesos inventados para ilustrar; no son la salida de un modelo real.)*

**⚠️ Cuándo no.** Si BM25 + densa + reranking ya cumplen tu SLO, no hace falta nada de esto. La interacción tardía multiplica el almacenamiento; SPLADE añade un modelo al indexado y a cada consulta; ColPali exige un modelo de visión y GPU para ir cómodo.

**🐞 Errores típicos.** (1) Cambiar de familia sin una medición con **tus** preguntas. (2) Olvidar el costo de almacenamiento del multivector. (3) Mezclar vectores de modelos distintos en un mismo índice.

### MaxSim a mano (gratis, sin modelo)

Dos vectores de 3 números por palabra, con ejes inventados (dinero, equipo, otros temas). La pregunta tiene dos palabras: *cuesta* y *router*. Comparamos un documento **corto** con las dos palabras relevantes y otro **largo** con esas dos más seis palabras de relleno.

In [ ]:
pregunta_tokens = np.array([[1.0, 0.1, 0.0],      # "cuesta"
                            [0.1, 1.0, 0.0]])     # "router"
relevantes = [[0.9, 0.2, 0.1],                    # "precio"
              [0.2, 0.9, 0.1]]                    # "módem"
relleno = [[0.0, 0.0, 1.0]] * 6                   # seis palabras de otros temas
doc_corto = np.array(relevantes)
doc_largo = np.array(relevantes + relleno)

def coseno(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"{'documento':<10}{'palabras':>9}{'MaxSim':>9}{'coseno de UN vector (promedio)':>34}")
for nombre, doc in (("corto", doc_corto), ("largo", doc_largo)):
    puntaje, _ = maxsim(pregunta_tokens, doc)
    print(f"{nombre:<10}{len(doc):>9}{puntaje:>9.2f}{coseno(pregunta_tokens.mean(axis=0), doc.mean(axis=0)):>34.2f}")

El documento largo tiene **las mismas dos palabras relevantes**, así que MaxSim da el mismo puntaje; en cambio, un **único** vector (el promedio) se **diluye** con el relleno y el coseno cae. Esa es la ventaja de la interacción tardía... y su precio: guardar **un vector por palabra**. Calculémoslo con el manual:

In [ ]:
palabras_manual = sum(len(f.texto.split()) for f in frs_base)
n_frag = len(frs_base)
un_vector = n_frag * 768 * 4                       # 768 números de 4 bytes por fragmento
multivector = palabras_manual * 128 * 4            # estimación: ≈ un vector de 128 números por palabra (como ColBERT)
print(f"{n_frag} fragmentos, {palabras_manual} palabras")
print(f"  un vector por fragmento (768 × 4 bytes) : {un_vector / 1024:>8.0f} KB")
print(f"  un vector por palabra   (128 × 4 bytes) : {multivector / 1024:>8.0f} KB   (≈ {multivector / un_vector:.1f} veces más)")
print("Es una ESTIMACIÓN aritmética (no se midió con un modelo ColBERT): sirve para ver el orden de magnitud.")

## Elegir embeddings y optimizarlos

**🧠 Qué es.** Tres decisiones sobre los vectores que ya usamos: **qué modelo**, **cuántas dimensiones** y **cuántos bits por número**.

**⚙️ Qué hace cada una.**
- **El modelo** fija la calidad y el idioma. Un modelo centrado en inglés recupera peor en español (lo vimos en la Sesión 4); por eso el curso usa `nomic-embed-text-v2-moe`, multilingüe, de **768 dimensiones**.
- **Matryoshka (MRL).** Algunos modelos se entrenan para que los **primeros** números del vector ya sean un buen vector pequeño, como las muñecas rusas. Se **trunca** a 256 o 128 y se **renormaliza** (se divide entre la nueva longitud, para que el coseno siga siendo un coseno). El modelo del curso declara soporte hasta 256 dimensiones: **lo verificamos con nuestras preguntas**.
- **Cuantización.** Guardar cada número con menos bits: **int8** (1 byte en vez de 4: 4 veces menos) o **1 bit** (solo el signo: 32 veces menos). Se pierde algo de precisión numérica.

**🎯 Para qué sirve.** Con millones de fragmentos el vector es lo que más pesa: memoria, disco, tiempo de búsqueda y costo de la base vectorial. Con 26 fragmentos da igual; la medición sirve para saber **cuánto se puede apretar sin perder recuperación**.

**🏗️ Analogía de infraestructura.** Es la **compresión de logs o de imágenes de contenedor**: eliges un nivel y vigilas que lo comprimido siga siendo útil. Matryoshka es bajar la resolución de un video; la cuantización es reducir la profundidad de color.

**✏️ Ejemplo a mano.** Un vector de 768 números en `float32` ocupa 768 × 4 = **3 072 bytes**. Truncado a 256: 1 024 bytes. Truncado a 256 y en `int8`: **256 bytes** (12 veces menos). Con un millón de fragmentos: 3.07 GB frente a 0.26 GB.

**⚠️ Cuándo no.** Si tu corpus es pequeño, no te molestes: gana complejidad sin ahorro real. Y **no todos los modelos se truncan igual**: uno que no se entrenó con Matryoshka pierde mucho más. Tampoco confundas *vector más corto* con *índice aproximado* (HNSW): son optimizaciones distintas y se pueden sumar.

**🐞 Errores típicos.** (1) Truncar **sin renormalizar**. (2) Truncar solo los fragmentos y no la pregunta (¡deben tener las mismas dimensiones!). (3) Decidir con 20 preguntas de otro dominio. (4) Mezclar en un mismo índice vectores de distinto modelo o distintas dimensiones.

### 1. ¿Cuánto cambia la recuperación con otro modelo?

Medimos los modelos de embeddings que tenga tu LM Studio, con los mismos 26 fragmentos y las mismas preguntas del golden set que tienen respuesta en el manual (20). Si solo tienes uno, verás una fila.

In [ ]:
TEXTOS_BASE = [texto_a_vectorizar(f) for f in frs_base]
modelos = modelos_de_embeddings_locales() if proveedor_embeddings() == "local" else [modelo_embeddings()]
VECTORES = {}                                                                # modelo → (matriz de fragmentos, consultas)
print(f"{'modelo':<44}{'dims':>6}{'recall':>8}{'precisión':>10}{'MRR':>6}{'1.º':>7}")
for modelo in modelos:
    try:
        VECTORES[modelo] = vectores_con(modelo, TEXTOS_BASE, GOLDEN)
    except Exception as exc:                                                 # un modelo que LM Studio no pudo cargar
        print(f"{modelo:<44}  (omitido: {type(exc).__name__})")
        continue
    m, q = VECTORES[modelo]
    r = evaluar_matriz(frs_base, puntajes(m, q), GOLDEN, k=4)
    print(f"{modelo:<44}{m.shape[1]:>6}{r['recall']:>8.0%}{r['precision']:>10.0%}{r['mrr']:>6.2f}{r['hit1']:>4}/{r['n']}")

Un modelo no pensado para español (los de la lista que son centrados en inglés) **pierde varios puntos** de recall y MRR sobre este manual. Elegir el modelo es la decisión de embeddings que más pesa, y se toma con **tus** preguntas en **tu** idioma.

### 2. Matryoshka y cuantización con el modelo del curso

Una sola tabla con todas las variantes del modelo del curso. En cada fila se **truncan los fragmentos y las preguntas** a las mismas dimensiones y se renormalizan:

In [ ]:
m, q = VECTORES[modelo_embeddings()]
completa = m.shape[1]

def fila(nombre, puntajes_por_pregunta, bytes_por_vector):
    r = evaluar_matriz(frs_base, puntajes_por_pregunta, GOLDEN, k=4)
    print(f"{nombre:<26}{bytes_por_vector:>9,}{completa * 4 / bytes_por_vector:>8.0f}×{r['recall']:>8.0%}{r['precision']:>10.0%}{r['mrr']:>6.2f}{r['hit1']:>4}/{r['n']}")

print(f"{'variante':<26}{'bytes/vec':>9}{'ahorro':>9}{'recall':>8}{'precisión':>10}{'MRR':>6}{'1.º':>7}")
fila(f"{completa} dims (float32)", puntajes(m, q), completa * 4)
for dims in (512, 256, 128, 64):
    fila(f"Matryoshka {dims} dims", puntajes(m, q, dims), dims * 4)

q8, esc = cuantizar_int8(m)                                                 # int8 sobre el vector completo
fila(f"{completa} dims int8", puntajes(descuantizar_int8(q8, esc), q), completa)
q8_256, esc_256 = cuantizar_int8(truncar(m, 256))                           # 256 dims y además int8
fila("256 dims + int8", puntajes(descuantizar_int8(q8_256, esc_256), {i: v[:256] for i, v in q.items()}), 256)

bits = cuantizar_binario(m)                                                 # 1 bit por dimensión
fila(f"{completa} dims 1 bit", {i: puntajes_hamming(cuantizar_binario(v[None, :]), bits).astype(float) for i, v in q.items()}, completa // 8)

### Cómo leer la tabla

- Mira **recall, MRR y 1.º a la vez**; la precisión baja un poco al acortar porque cambia el orden de los fragmentos del top-4, y con 20 preguntas **1 pregunta ≈ 5 puntos de hit@1** (ruido).
- **Matryoshka cumple lo que declara el modelo**: hasta 256 dimensiones la recuperación se mantiene casi intacta con **3 veces menos** bytes; la caída aparece más abajo. Es una afirmación sobre **este corpus y estas preguntas**: con otro, se vuelve a medir.
- **int8 no cuesta nada medible** aquí y ahorra 4 veces. Combinado con 256 dimensiones, 12 veces. **1 bit** ahorra 32 veces y ya se nota en el MRR.
- Ojo con lo que **no** mide esta tabla: la velocidad de la búsqueda y el efecto en un índice aproximado (HNSW) con millones de vectores.

### 3. ¿Y los otros modelos? ¿Todos se truncan igual?

Misma prueba de truncado con cada modelo que midió la celda 1 (solo recall y MRR):

In [ ]:
print(f"{'modelo':<44}" + "".join(f"{d:>13}" for d in ("completo", "256 dims", "128 dims")))
for modelo, (m_x, q_x) in VECTORES.items():
    celdas = []
    for dims in (None, 256, 128):
        r = evaluar_matriz(frs_base, puntajes(m_x, q_x, dims), GOLDEN, k=4)
        celdas.append(f"{r['recall']:.0%}/{r['mrr']:.2f}")
    print(f"{modelo:<44}" + "".join(f"{c:>13}" for c in celdas))
print("\n(cada celda: recall / MRR)")

Un modelo se trunca bien **si se entrenó para eso**; en los otros, cada recorte cuesta más. No lo asumas por el nombre: **mídelo con tus preguntas**.

### 4. Cuánto pesaría con un millón de fragmentos (aritmética, gratis)

In [ ]:
N = 1_000_000
variantes_tamano = {
    "768 dims float32": 768 * 4,
    "768 dims int8": 768,
    "256 dims float32 (Matryoshka)": 256 * 4,
    "256 dims int8": 256,
    "768 dims 1 bit": 768 // 8,
}
for nombre, bytes_vector in variantes_tamano.items():
    print(f"{nombre:<32}{N * bytes_vector / 1e9:>7.2f} GB")
print("\nSolo los vectores: sin metadatos ni el índice de búsqueda aproximada.")

> 🧰 **Regla práctica.** Primero elige el **modelo** con tus preguntas; después, si el tamaño duele, prueba **Matryoshka** y **int8** (casi gratis en calidad); deja el **1 bit** para cuando el almacenamiento sea el problema y tengas margen para reordenar después. Y cada vez que cambies de modelo o de dimensiones, **reindexa todo**.

---
# Parte 8 · RAG multimodal: audio y video

Hasta ahora todo era **texto**. Pero mucho conocimiento de una empresa vive en **reuniones grabadas, capacitaciones, llamadas de soporte y videos**. La buena noticia: un RAG de audio o de video es **el mismo RAG**, con un paso de ingesta distinto. La mala: ese paso nuevo tiene errores propios y hay que medirlo, como todo lo demás.

## El pipeline estándar

```
 audio / video ──► ffmpeg (solo el audio, 16 kHz, mono) ──► Whisper (ASR) ──► segmentos {inicio, fin, texto}
                                                                                      │
        ventanas de ≈ 50 s (con solape) ◄──────────────────────────────────────────────┘
               │   cada fragmento recuerda su inicio y su fin
               ▼
      embeddings (el embeddings.py del curso) ──► almacén ──► búsqueda ──► LLM ──► respuesta con cita  [03:00–03:55]

 video (opcional) ──► fotogramas clave (ffmpeg) ──► descripción con un modelo con visión / OCR ──┐
                                                       alineada por TIEMPO con la transcripción ◄─┘
```

**🧠 Qué es.** **ASR** (*automatic speech recognition*) convierte voz en texto; **Whisper** es un modelo abierto de ASR, y `faster-whisper` es una implementación rápida que corre en CPU. El resultado no es un bloque de texto: son **segmentos con `inicio` y `fin`** en segundos. Esos tiempos son el tesoro: permiten **citar** (`mm:ss`) y que una persona vaya a verificar en el reproductor.

**⚙️ Qué hace cada paso.** (1) `ffmpeg` quita la imagen y deja el audio. (2) Whisper transcribe y devuelve segmentos. (3) Se **juntan segmentos en ventanas** de ≈ 45 a 60 segundos (con un poco de solape) para tener fragmentos del tamaño que el RAG ya conoce. (4) Se vectorizan con los embeddings de siempre. (5) La respuesta cita el **rango de tiempo** de los fragmentos que usó.

**🎯 Para qué sirve.** Preguntarle a una charla, una reunión o un curso: "¿qué se acordó sobre el presupuesto?" y recibir la respuesta **con el minuto donde se dijo**.

**🏗️ Analogía de infraestructura.** Es el **pipeline de logs con marca de tiempo**: lo valioso de una línea de log no es solo su texto sino su `timestamp`. Perderlo es dejar de poder correlacionar. Y la transcripción automática es el **parser**: si falla, todo lo que sigue hereda el error (igual que con el OCR de la Sesión 6).

**✏️ Ejemplo a mano.** Segmentos de 6 a 8 segundos cada uno. Una ventana de 50 s junta unos 7 segmentos, por ejemplo de `03:00` a `03:52`, y su cita sale como `[03:00–03:52]`.

## ¿Qué indexar? Cuatro enfoques

| Enfoque | Qué se vectoriza | Cuándo conviene | Limitación |
| --- | --- | --- | --- |
| **Solo transcripción** | El texto del habla, con tiempos | Charlas, reuniones, podcasts, llamadas: la información está **dicha** | Pierde lo que solo se **ve** (diagramas, diapositivas, notas) |
| **Transcripción + fotogramas** | Habla + descripción u OCR de fotogramas clave, alineados por tiempo | Video donde lo escrito o mostrado importa (diapositivas, demos, pizarra) | Los modelos locales con visión son **lentos y fallan** con letra manuscrita o baja resolución |
| **OCR de fotogramas** | El texto que aparece en pantalla | Capturas de pantalla, diapositivas con texto claro | Mal con manuscrita; ruido de interfaces |
| **Embeddings de la imagen** (**ColPali / ColQwen**) | La **imagen de la página** (un vector por trozo de imagen), con interacción tardía | **Documentos visuales** (PDF con tablas, gráficos, diseño) sin pasar por OCR | Modelo de visión y GPU; mucho más almacenamiento. **Hoy se explica, no se implementa** |

**⚠️ Limitaciones honestas (medidas con los medios de esta clase).**
- **La transcripción es lo que sostiene el RAG.** En un video de notas escritas a mano en un cristal, un modelo local de 12B con visión a 640×360 lee la letra de forma **irregular** y tarda unos 15 segundos por fotograma. En el fotograma que probamos leyó bien "AGENTS!" y "RISK", pero convirtió "↑ 0-CLICKS" en "↓ CLICKS" (la flecha invertida **cambia el sentido**) y "BY" en "SY"; en otra prueba leyó "AGENTS!" como "PACIENTS". La voz, en cambio, tenía casi toda la información.
- **El ASR se equivoca con el vocabulario técnico.** El modelo `small` escribió "chatGBT" donde la charla dice "ChatGPT". Se mitiga con `initial_prompt` (le das el vocabulario que debe esperar), pero **no lo arregla todo**: hay que **revisar** una muestra de la transcripción, igual que revisaste el texto extraído de un PDF.
- **El idioma:** con medios en inglés, las preguntas en español funcionan porque el modelo de embeddings es **multilingüe**; pero el LLM debe responder en español sobre fragmentos en inglés. Se mide, no se supone.

**🐞 Errores típicos.** (1) Perder los tiempos al fragmentar. (2) Fragmentos de 5 segundos (sin contexto) o de 5 minutos (vector borroso). (3) No revisar la transcripción. (4) Transcribir cada vez que se reinicia el kernel (¡se guarda en caché!). (5) Subir al repositorio medios de terceros.

> 📦 **Requisitos (opcionales).** `uv sync --group media` (instala `faster-whisper` y `av<16`; **no hace falta torch**), `ffmpeg` en el sistema y los medios en `datos/media/` (carpeta ignorada por Git; el instructor los entrega). La primera ejecución descarga el modelo de Whisper (`small` ≈ 460 MB). Si algo falta, las celdas siguientes lo **avisan** y se omiten.
>
> 🔗 Hay una **implementación formal para estudiar** de este pipeline en el proyecto `rag-playground/rag-intermedio` del repositorio del instructor. Aquí no dependemos de ella.

In [ ]:
HAY_WHISPER, msg_whisper = hay_faster_whisper()
MEDIOS = medios_disponibles()
print(("✅ " if HAY_WHISPER else "⚪ ") + msg_whisper)
print(("✅ ffmpeg disponible" if hay_ffmpeg() else "⚪ ffmpeg no encontrado (solo hace falta para extraer audio o fotogramas)"))
if MEDIOS:
    for nombre, ruta in MEDIOS.items():
        print(f"  • {nombre} ({ruta.stat().st_size / 1e6:.1f} MB)")
else:
    print("⚪ No hay medios en datos/media/. Pide al instructor el MP3 y el MP4 y colócalos ahí; mientras tanto, lee esta parte sin ejecutarla.")
HAY_MEDIA = HAY_WHISPER and bool(MEDIOS)
MODELO_WHISPER = "small"          # "base" es ~3 veces más rápido; "small" comete menos errores
IDIOMA_MEDIA = "en"               # los medios de la clase están en inglés; None = que Whisper lo detecte

### 1. Transcribir: del sonido a segmentos con tiempo

La primera vez tarda (la charla dura unos 12 minutos); el resultado se guarda en `datos/media/.cache/`, así que reiniciar el kernel no obliga a transcribir de nuevo.

In [ ]:
TRANSCRIPCIONES = {}
if HAY_MEDIA:
    for nombre, ruta in MEDIOS.items():
        t = transcribir(ruta, modelo=MODELO_WHISPER, idioma=IDIOMA_MEDIA)
        TRANSCRIPCIONES[nombre] = t
        origen = "desde caché (la primera vez tardó %.0f s)" % t.segundos if t.desde_cache else "transcrito ahora"
        print(f"{nombre[:55]:<56} {mmss(t.duracion)} de audio · {t.segundos:>3.0f} s ({t.veces_tiempo_real:.0f}× tiempo real) · "
              f"{len(t.segmentos)} segmentos · {t.palabras} palabras · {origen}")
    nombre_audio = next((n for n in MEDIOS if n.lower().endswith(".mp3")), next(iter(MEDIOS)))
    print(f"\nLos primeros segmentos de «{nombre_audio[:40]}…»:")
    for s in TRANSCRIPCIONES[nombre_audio].segmentos[:5]:
        print(f"  [{mmss(s.inicio)} → {mmss(s.fin)}] {s.texto}")
else:
    print("⚪ Parte omitida: faltan la dependencia o los medios (ver la celda anterior).")

### 2. La transcripción tiene errores: revísala

Buscamos en el texto los términos técnicos que sabemos que aparecen ("ChatGPT", "Copilot"). Y probamos la mitigación: darle a Whisper el vocabulario con `initial_prompt`.

In [ ]:
import re

def terminos(t, patron):
    return re.findall(patron, " ".join(s.texto for s in t.segmentos), flags=re.I)

if HAY_MEDIA and nombre_audio in TRANSCRIPCIONES:
    VOCABULARIO = "Vocabulario: ChatGPT, GitHub Copilot, GitHub, LLM, RAG, Generative AI, Python, Tinder."
    con_prompt = transcribir(MEDIOS[nombre_audio], modelo=MODELO_WHISPER, idioma=IDIOMA_MEDIA, initial_prompt=VOCABULARIO)
    sin_prompt = TRANSCRIPCIONES[nombre_audio]
    for etiqueta, t in (("sin initial_prompt", sin_prompt), ("con initial_prompt", con_prompt)):
        print(f"{etiqueta:<20} 'ChatGPT' → {terminos(t, r'chat ?g[a-z]{2}')}   'Copilot' → {terminos(t, r'co-?pilot')}   'RAG' → {terminos(t, r'rack|rag')}")
else:
    print("⚪ Parte omitida: faltan la dependencia o los medios.")

**Lee la diferencia:** el vocabulario corrige los nombres propios que le diste ("ChatGPT"), pero **no** lo que no previste ni todo lo que sí previste ("RAG" sigue saliendo como otra palabra). Es una mitigación, no una garantía: por eso se **revisa una muestra** y se corrige a mano lo crítico (nombres, cifras) antes de indexar.

### 3. Fragmentar con tiempo e indexar

Ventanas de ≈ 50 segundos con 10 de solape. Cada fragmento lleva su `inicio` y su `fin`; la «sección» del fragmento es su rango de tiempo, y eso es lo que usaremos como cita.

In [ ]:
ALMACENES_MEDIA = {}
if HAY_MEDIA:
    for nombre, t in TRANSCRIPCIONES.items():
        ventanas = ventanas_de_tiempo(t.segmentos, ventana_s=50, solape_s=10)
        fr = a_fragmentos(ventanas, nombre)
        vectores = vectorizar([f.texto for f in fr], tipo="documento")
        for f, v in zip(fr, vectores): f.vector = v
        almacen_m = crear_almacen("memoria"); almacen_m.preparar(len(vectores[0])); almacen_m.insertar(fr)
        ALMACENES_MEDIA[nombre] = almacen_m
        print(f"{nombre[:55]:<56} {len(fr):>3} fragmentos · {sum(len(f.texto.split()) for f in fr) / len(fr):.0f} palabras de media")
    primero = next(iter(ALMACENES_MEDIA.values()))._fragmentos[2]
    print(f"\nUn fragmento: {cita_de_tiempo(primero)}\n   {primero.texto[:230]}…")
else:
    print("⚪ Parte omitida: faltan la dependencia o los medios.")

### 4. Preguntar EN ESPAÑOL sobre contenido en INGLÉS (y medirlo)

`datos/preguntas_media.json` trae 16 preguntas en español (8 por medio) con el **tramo de tiempo** donde está la respuesta. Un acierto es que alguno de los fragmentos recuperados **se solape con ese tramo**. Es la métrica de siempre (hit@k y MRR), pero con **tiempo** en lugar de secciones.

In [ ]:
PREGUNTAS_MEDIA = json.loads(Path("datos/preguntas_media.json").read_text(encoding="utf-8"))["preguntas"]
RESULTADOS_MEDIA = {}
if HAY_MEDIA:
    for nombre, almacen_m in ALMACENES_MEDIA.items():
        preguntas_m = [p for p in PREGUNTAS_MEDIA if p["archivo"] == nombre]
        if not preguntas_m:
            continue
        r = evaluar_por_tiempo(lambda texto, k, a=almacen_m: a.buscar(vector_consulta(texto), k=k), preguntas_m, ks=(1, 3))
        RESULTADOS_MEDIA[nombre] = r
        print(f"{nombre[:55]}\n   hit@1 {r['hit@1']}/{r['n']} · hit@3 {r['hit@3']}/{r['n']} · MRR {r['mrr']:.2f}")
        for f in r["filas"]:
            marca = "✅" if f["posicion"] == 1 else ("🟡" if f["posicion"] else "❌")
            print(f"   {marca} {f['id']} puesto {f['posicion'] or '—':<2} · esperado {f['esperado']} · el 1.º llegó de {f['primero']} · {f['pregunta'][:62]}")
    total = sum(r["n"] for r in RESULTADOS_MEDIA.values()) or 1
    print(f"\nEn conjunto: hit@1 {sum(r['hit@1'] for r in RESULTADOS_MEDIA.values())}/{total} · "
          f"hit@3 {sum(r['hit@3'] for r in RESULTADOS_MEDIA.values())}/{total}")
else:
    print("⚪ Parte omitida: faltan la dependencia o los medios.")

### Cómo leer este resultado

- 🟡 y ❌ son las preguntas que merecen mirarse **una por una**: ¿la palabra clave de la respuesta se escribió mal en la transcripción?, ¿la idea está repartida entre dos ventanas?, ¿la pregunta usa una palabra ("confesión") que en el original es otra ("confession time")?
- Ya hay un hallazgo útil: **el modelo de embeddings multilingüe permite preguntar en español sobre inglés** y recupera el tramo correcto en la gran mayoría de los casos, sin traducir nada.
- Con 16 preguntas, **una de diferencia ≈ 6 puntos**: es ruido. Sirve para comparar configuraciones, no para prometer exactitud.

### ¿Importa el tamaño de la ventana?

Misma prueba con ventanas de 30, 50 y 90 segundos. Ojo: una ventana más larga tiene **más probabilidad de solaparse** con el tramo esperado, así que esta métrica la favorece; mira también las palabras por ventana (el costo en contexto).

In [ ]:
if HAY_MEDIA:
    print(f"{'ventana':>9}{'palabras/ventana':>18}{'hit@1':>8}{'hit@3':>8}{'MRR':>7}")
    for ventana_s in (30, 50, 90):
        h1 = h3 = n = 0; mrr = 0.0; palabras = []
        for nombre, t in TRANSCRIPCIONES.items():
            preguntas_m = [p for p in PREGUNTAS_MEDIA if p["archivo"] == nombre]
            fr = a_fragmentos(ventanas_de_tiempo(t.segmentos, ventana_s, 10), nombre)
            vectores = vectorizar([f.texto for f in fr], tipo="documento")
            for f, v in zip(fr, vectores): f.vector = v
            a = crear_almacen("memoria"); a.preparar(len(vectores[0])); a.insertar(fr)
            r = evaluar_por_tiempo(lambda texto, k, a=a: a.buscar(vector_consulta(texto), k=k), preguntas_m, ks=(1, 3))
            h1 += r["hit@1"]; h3 += r["hit@3"]; n += r["n"]; mrr += r["mrr"] * r["n"]
            palabras += [len(f.texto.split()) for f in fr]
        print(f"{ventana_s:>7} s{sum(palabras) / len(palabras):>18.0f}{h1:>5}/{n}{h3:>5}/{n}{mrr / n:>7.2f}")
else:
    print("⚪ Parte omitida: faltan la dependencia o los medios.")

### 5. Responder con cita de tiempo (💰 o gratis con modelo local)

El prompt `rag_media/v2` le pide al modelo responder **en español** usando solo la transcripción y **citar el rango de tiempo** de cada dato. Después **comprobamos por código** que cada cita apunta a un fragmento que realmente se recuperó (la misma idea de "citas válidas" de la Parte 9). Y, como el modelo puede no citar, el sistema añade siempre una línea de **fuentes** con los tramos recuperados: la verificabilidad no debe depender de que el modelo obedezca.

> 🐞 **Lo que aprendimos al probarlo:** con la primera versión del prompt (`v1`), el modelo local de 3B citaba el tramo solo en algunas respuestas y en una pregunta se abstuvo aunque el tramo correcto estaba entre los recuperados. Repetir la instrucción de citar **al final** del prompt (`v2`) lo arregló en las preguntas probadas. Cambió una línea del prompt, y solo lo vimos porque medimos.

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.0015 con un modelo de proveedor: 3 preguntas; gratis con el modelo local)
def citas_de_tiempo(texto):
    # Rangos 'mm:ss–mm:ss' que aparecen en la respuesta.
    return re.findall(r"(\d{1,2}:\d{2})\s*[–-]\s*(\d{1,2}:\d{2})", texto)

if PUEDE_GASTAR and HAY_MEDIA:
    for id_ in ("T1", "V2", "V3"):
        p = next((x for x in PREGUNTAS_MEDIA if x["id"] == id_ and x["archivo"] in ALMACENES_MEDIA), None)
        if p is None:
            continue
        top = ALMACENES_MEDIA[p["archivo"]].buscar(vector_consulta(p["pregunta"]), k=3)
        r = gasto.registrar(generar(renderizar("rag_media", "v2", fragmentos=top, pregunta=p["pregunta"]), max_tokens=300))
        rangos = {f.metadatos["seccion"] for f in top}
        citas = citas_de_tiempo(r.texto)
        validas = sum(f"{a}–{b}" in rangos for a, b in citas)
        print(f"{p['id']} · {p['pregunta']}\n   → {r.texto.strip()[:280]}\n   citas de tiempo del modelo: {len(citas)} ({validas} válidas) · "
              f"fuentes añadidas por el sistema: {' '.join(cita_de_tiempo(f) for f in top[:2])}\n   esperado: {p['respuesta_esperada']}\n")
else:
    print("⚪ Celda de pago desactivada, o faltan la dependencia o los medios.")

### 6. Video = audio + fotogramas clave (opcional)

Para el video, el **audio** ya está cubierto: es la misma transcripción. Lo que añade el video es la imagen. Con `ffmpeg` se sacan **fotogramas clave** (uno cada N segundos) y se **alinean por tiempo**: para un fragmento de `08:29–08:52` se busca el fotograma más cercano a ese rango y se podría describir con un modelo con visión o con OCR, y guardar la descripción junto al fragmento.

In [ ]:
nombre_video = next((n for n in MEDIOS if n.lower().endswith((".mp4", ".mkv", ".mov", ".webm"))), None)
if HAY_MEDIA and nombre_video and hay_ffmpeg():
    t0 = time.perf_counter()
    wav = extraer_audio(MEDIOS[nombre_video], MEDIOS[nombre_video].parent / ".cache" / "audio_16k.wav")
    print(f"ffmpeg → audio mono a 16 kHz: {wav.stat().st_size / 1e6:.0f} MB en {time.perf_counter() - t0:.1f} s")
    t0 = time.perf_counter()
    fotogramas = extraer_fotogramas(MEDIOS[nombre_video], cada_s=120)
    print(f"ffmpeg → {len(fotogramas)} fotogramas (uno cada 120 s) en {time.perf_counter() - t0:.1f} s")
    segundo = 509                                                  # el tramo de la pregunta V4 (cero clics)
    t_f, ruta_f = fotograma_mas_cercano(fotogramas, segundo)
    print(f"El fotograma más cercano al segundo {segundo} ({mmss(segundo)}) es el de {mmss(t_f)}: {ruta_f.name}")
else:
    print("⚪ Parte omitida: faltan el video, la dependencia o ffmpeg.")

In [ ]:
# Opcional y LENTO: describir un fotograma con un modelo con visión de LM Studio (≈ 16 s por fotograma con un 12B en CPU).
DESCRIBIR_FOTOGRAMAS = False          # ← ponlo en True solo si quieres verlo; gratis, pero tarda y gasta tokens de razonamiento
if DESCRIBIR_FOTOGRAMAS and HAY_MEDIA and nombre_video and hay_ffmpeg():
    t0 = time.perf_counter()
    print(describir_fotograma(ruta_f))
    print(f"\n({time.perf_counter() - t0:.0f} s) Compara con lo que ves en la imagen: ¿leyó bien la escritura a mano?")
else:
    print("⚪ Descripción con visión desactivada (DESCRIBIR_FOTOGRAMAS = False).")

### Lo que enseña esta parte

1. **Un RAG de audio es un RAG de texto con un parser nuevo**: lo que importa es no perder los tiempos y revisar la transcripción.
2. **Citar con `mm:ss` convierte una respuesta en algo verificable**: quien duda, va al minuto y escucha.
3. **En video, la transcripción es lo que sostiene el RAG**; los fotogramas son un complemento que hay que justificar con una medición (y los modelos locales con visión son lentos y débiles con escritura manuscrita).
4. **Los documentos visuales** (PDF con tablas y gráficos) tienen su propia familia: **ColPali / ColQwen** vectorizan la imagen de la página con interacción tardía. No los implementamos, pero ya sabes cuándo mirarlos: cuando el OCR o la extracción de tablas te cuestan respuestas medidas.
5. **La implementación formal** de este pipeline (paquete, pruebas, configuración) está en `rag-playground/rag-intermedio` del repositorio del instructor: úsala para estudiar, no la necesitas para esta clase.

---
# Parte 9 · Evaluar la generación y diagnosticar por capa

Volvemos al manual de Andes Fibra y al golden set de 24 preguntas. Hasta ahora medimos solo la **recuperación**. Ahora medimos la respuesta del modelo con **comprobaciones por código** (baratas, deterministas, repetibles):

| Comprobación | Qué detecta | Cómo |
| --- | --- | --- |
| **Dato correcto** | La respuesta trae el dato clave esperado | `claves` del golden set aparecen en el texto |
| **Citas válidas** | El modelo cita fragmentos que existen | `[n]` dentro de los fragmentos entregados |
| **"No sé" bien dicho** | Se abstiene en las preguntas fuera de alcance | La frase exacta del prompt |
| **Cifras con respaldo** | Una cifra de la respuesta que no está en el contexto (un *checksum* de la alucinación) | Números de la respuesta ⊆ números del contexto ∪ pregunta |

### El diagnóstico por capa

La columna más útil de un informe: **¿en qué capa falló cada pregunta?** Para cada una, la primera categoría que aplique:

| Diagnóstico | Capa | Significa |
| --- | --- | --- |
| `abstencion_por_recuperacion` / `recuperacion_incompleta` | **recuperación** | La información necesaria no llegó al prompt; el modelo hizo lo que pudo |
| `abstencion_indebida` | generación | Llegó todo, pero el modelo dijo "no sé" |
| `dato_incorrecto` | generación | Llegó todo, pero el dato no está en la respuesta |
| `cifra_sin_respaldo` / `alucinacion` | generación (**alucinación**) | Inventó una cifra, o contestó algo que no se podía contestar |
| `sin_citas_validas` | verificabilidad | Respondió bien, pero sin citar o citando mal |

💰 **FYI:** 24 llamadas al LLM (≈ US$ 0.012 con un modelo de proveedor; gratis con el modelo local).

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.012 con un modelo de proveedor; gratis con el modelo local)
if PUEDE_GASTAR:
    inf_v1 = correr("v1 · k=4", base, k=4, version="v1")
    imprimir_resumen(inf_v1, "RAG de la Sesión 5")
    print("\n" + tabla_por_tipo(inf_v1))
else:
    print("⚪ Celda de pago desactivada.")

In [ ]:
if PUEDE_GASTAR:
    print("Dónde falló cada pregunta:\n")
    for diag, ids in inf_v1.diagnosticos().items():
        print(f"{diag:<30} capa: {CAPA_DEL_DIAGNOSTICO[diag]:<26} {', '.join(ids)}")

    print("\nUn ejemplo de cada diagnóstico:\n")
    mostrados = set()
    for f in inf_v1.filas:
        if f.diagnostico not in ("ok", *mostrados):
            mostrados.add(f.diagnostico)
            print(f"[{f.id} · {f.diagnostico}] {f.pregunta}\n     → {f.respuesta[:210]!r}\n")

### Cómo leer este informe

1. **Mira la capa antes que el porcentaje.** Si los fallos son de *recuperación*, cambia el recuperador (Partes 3 a 8); si son de *generación*, cambia el prompt o el modelo. Tocar la capa equivocada no mueve nada.
2. **"0 alucinaciones" puede esconder otro problema.** Un modelo que dice "no sé" a todo no alucina nunca... y no sirve. Mira siempre **juntas** las dos columnas: alucinaciones y *"no sé" indebido*. Bajar una a costa de subir la otra no es mejorar.
3. **Las comprobaciones por código se equivocan, y hay que conocer cómo.** Una respuesta como *"No, no es normal."* (para el fraude del SMS) es correcta pero no contiene la palabra clave `fraude`: la marcamos mal (**falso negativo**). Lo contrario también pasa: la cifra de un concepto equivocado (el 15 del vencimiento usado como día de suspensión) **sí está en el contexto**, así que el chequeo de cifras no la ve. Las comprobaciones por código son el piso, no el techo: por eso existe el juez de la Parte 11.
4. **Con un modelo local de 3B parámetros** verás más abstenciones indebidas y citas flojas que con un modelo de proveedor. No es un defecto de tu código: es el modelo. Cambia `RAG_PROVIDER` y compara.

> ⚙️ **Ruido de ejecución.** Aun con temperatura 0, un servidor local puede dar respuestas ligeramente distintas entre corridas. Si una pregunta cambia de diagnóstico al repetir, es ruido: por eso comparamos con **muchas preguntas** y miramos diferencias grandes.

---
# Parte 10 · Alucinaciones: causas, mitigación y regresión

Una **alucinación** es una respuesta plausible pero no respaldada (o falsa). En un RAG casi nunca es "el modelo se volvió loco": tiene **causas identificables**, y cada una tiene su mitigación.

| Causa | Cómo se ve | Mitigación |
| --- | --- | --- |
| **Recuperación pobre** | El fragmento correcto no llegó; el modelo rellena | Mejorar recuperación; **abstenerse** |
| **El prompt no permite el "no sé"** | Contesta siempre, aunque no sepa | Frase exacta de abstención; probar con preguntas fuera de alcance |
| **Pregunta "parecida pero ausente"** | El tema suena al manual, el dato no está | "No sé" explícito para temas parecidos (prompt v3); evaluarlo |
| **Contexto contradictorio u obsoleto** | El modelo **cita fielmente** un dato viejo | **Higiene de la base**: versiones, vigencia, metadatos y filtros |
| **El modelo ignora el contexto** | Usa su "memoria" y no lo recuperado | Modelo más capaz; verificar con juez y con cifras |
| **Extrapolar o deducir** | "Como el de 600 cuesta 55, el de 1 Gbps costaría ~90" | Prohibir estimar; verificar cifras |

## Demo 1 · Un prompt sin salida de escape
Dos prompts con las mismas preguntas "trampa" (fuera de alcance pero parecidas al manual): uno **ingenuo** (sin regla de "no sé") y el v3 del curso.

💰 **FYI:** 4 llamadas (≈ US$ 0.002 con un modelo de proveedor; gratis con el modelo local).

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.002 con un modelo de proveedor; gratis con el modelo local)
def prompt_ingenuo(pregunta, fragmentos):
    # Sin regla de "no sé", sin citas, sin 'solo el contexto': el prompt de quien no ha medido nunca.
    ctx = "\n\n".join(f"[{i}] {f.texto}" for i, f in enumerate(fragmentos, start=1))
    return f"Responde la pregunta del cliente con este contexto.\n\n{ctx}\n\nPregunta: {pregunta}"

if PUEDE_GASTAR:
    for id_ in ("F02", "F01"):
        p = next(x for x in GOLDEN if x["id"] == id_)
        frs = recuperar(base, p["pregunta"], k=4)
        for etiqueta, prompt in (("ingenuo", prompt_ingenuo(p["pregunta"], frs)), ("v3", construir_prompt(p["pregunta"], frs, "v3"))):
            r = gasto.registrar(generar(prompt, max_tokens=300))
            print(f"[{id_} · {etiqueta}] {p['pregunta']}\n     → {r.texto[:260]!r}\n")
else:
    print("⚪ Celda de pago desactivada.")

Con el prompt ingenuo, el modelo suele **disculparse y luego deducir** ("podemos inferir que… es posible que…"): una alucinación con buenos modales. Con el v3, responde la frase exacta. **La diferencia entre las dos es una línea del prompt, y solo la ves si evalúas con preguntas fuera de alcance.**

## Demo 2 · Contexto obsoleto: fiel, pero incorrecto

Alguien dejó en la base una **tabla de precios anterior** (35 dólares en vez de 39). Se indexa junto al manual. ¿Qué pasa?

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.0005 con un modelo de proveedor; gratis con el modelo local)
VIEJO = Fragmento(
    id="tabla_vieja#0", metadatos={"fuente": "precios_2025.md", "seccion": "1. Planes de internet", "n": 0, "vigencia": "anterior"},
    texto="Tabla de precios anterior (vigente hasta el año pasado): el plan Familiar de 300 Mbps cuesta 35 dólares al mes.",
)
VIEJO.vector = vectorizar([texto_a_vectorizar(VIEJO)])[0]

contaminada = crear_almacen("memoria")
ingestar_con_metadatos(contaminada, MANUAL, {"vigencia": "actual"})
contaminada.insertar([VIEJO])

p = next(x for x in GOLDEN if x["id"] == "G01")
frs = recuperar(contaminada, p["pregunta"], k=4)
print("Top-4:", [(f.id, round(f.puntaje, 2)) for f in frs], "\n")

if PUEDE_GASTAR:
    r = gasto.registrar(generar(construir_prompt(p["pregunta"], frs, "v3"), max_tokens=300))
    print("Respuesta:", r.texto)
    print("\n¿Contiene el dato correcto (39)?       ", contiene_claves(r.texto, p["claves"]))
    print("Cifras sin respaldo en el contexto:     ", numeros_sin_respaldo(r.texto, frs, p["pregunta"]) or "ninguna  ← el chequeo de cifras NO la ve")

La respuesta **cita fielmente** un fragmento que está en el contexto: el juez de fidelidad la aprobaría y el chequeo de cifras también. Pero **es incorrecta**. Esto separa dos ideas que se confunden:

| | Pregunta | Se mide con |
| --- | --- | --- |
| **Fidelidad** (*groundedness*) | ¿Lo que dice está **en el contexto**? | Cifras con respaldo, juez de fidelidad |
| **Corrección** | ¿Lo que dice es **verdad**? | Respuesta esperada del golden set |

Un RAG puede ser **fiel e incorrecto** (contexto malo) o, más raro, **correcto e infiel** (acertó de memoria). Hay que medir las dos.

**La mitigación no está en el prompt, está en los datos:** guardar un metadato de **vigencia** y **filtrar** al buscar.

In [ ]:
filtrado = lambda pregunta, k: contaminada.buscar(vector_consulta(pregunta), k=k, filtro={"vigencia": "actual"})
frs_ok = filtrado(p["pregunta"], 4)
print("Top-4 con filtro de vigencia:", [(f.id, round(f.puntaje, 2)) for f in frs_ok])
if PUEDE_GASTAR:
    r = gasto.registrar(generar(construir_prompt(p["pregunta"], frs_ok, "v3"), max_tokens=300))
    print("Respuesta:", r.texto, "\n¿Contiene el dato correcto?", contiene_claves(r.texto, p["claves"]))

Es el mismo principio que **retirar una imagen de contenedor obsoleta del registro**: da igual lo bien que esté escrito el *deployment*, si el registro sirve la imagen vieja. (Y por eso el filtro por metadatos, que también sirve para permisos, se aplica **en la recuperación**, nunca confiando en el prompt.)

## Medir → cambiar → volver a medir

Ahora el ciclo completo, con el golden set de 24 preguntas y la generación. Cada fila cambia **una cosa** respecto a la anterior:

| Configuración | Qué cambia |
| --- | --- |
| `v1 · k=4` | El RAG de la Sesión 5 (ya medido) |
| `v2 · k=4` | Prompt v2: copiar cifras tal cual y respuestas cortas |
| `v3 · k=4` | Prompt v3: prohíbe estimar, extiende el "no sé" a lo parecido-ausente, cubre las interpretaciones de una pregunta ambigua |
| `s2b sección k=2 · v3` | Recuperación **small-to-big** (Parte 3) con el prompt v3 |
| `descomposición · v3` | Recuperación con **descomposición** (Parte 4) con el prompt v3 |

💰 **FYI:** 4 configuraciones nuevas × 24 preguntas = 96 llamadas (≈ US$ 0.05 con un modelo de proveedor, más ≈ 24 llamadas cortas de la descomposición; gratis con el modelo local). Tarda unos minutos con el local.

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.05 con un modelo de proveedor; gratis con el modelo local)
if PUEDE_GASTAR:
    t0 = time.perf_counter()
    s2b = RecuperadorSmallToBig(peq, MANUAL, frs_peq, modo="seccion")
    correr("v2 · k=4", base, k=4, version="v2")
    correr("v3 · k=4", base, k=4, version="v3")
    correr("s2b sección k=2 · v3", None, k=2, version="v3", recuperador=s2b)
    correr("descomposición · v3", None, k=4, version="v3", recuperador=RecuperadorDescompuesto(base, k_sub=3))
    ciclo = {n: INFORMES[n] for n in ("v1 · k=4", "v2 · k=4", "v3 · k=4", "s2b sección k=2 · v3", "descomposición · v3")}
    print(f"({time.perf_counter() - t0:.0f} s)\n")
    print(tabla_comparativa(ciclo, ("recall", "precision", "datos_correctos", "citas_validas", "abstencion_correcta",
                                    "abstencion_indebida", "alucinaciones", "ok")))
else:
    print("⚪ Celda de pago desactivada.")

### Cómo leer la tabla

- Mira **juntas** *datos correctos*, *"no sé" bien dicho*, *"no sé" indebido* y *alucinaciones*. Una mejora real sube las dos primeras sin disparar las otras.
- Cambiar el **prompt** mueve la **generación** (datos, citas, abstenciones) y deja intactos Recall y Precision. Cambiar el **recuperador** mueve Recall/Precision y, a través del contexto, la generación. **Cada cambio toca una capa**: por eso se mide por capas.
- Una diferencia de 1 pregunta (≈ 4 puntos) es ruido; una de 4 o 5 es señal.
- Puede ocurrir que **ninguna configuración sea la mejor en todo**. Eso es lo normal: elegir es decidir qué falla te cuesta más (¿una respuesta inventada o un "no sé" de más?).

## El SLO: ¿puedo desplegar este cambio?

Un SLO convierte el informe en una **decisión**. Los valores por defecto (en `evaluacion.SLO_POR_DEFECTO`) son solo un ejemplo: en un proyecto real los fija quien responde por el servicio.

In [ ]:
print("SLO:", {k: f"{op} {v:.0%}" for k, (op, v) in SLO_POR_DEFECTO.items()}, "\n")
for nombre, inf in INFORMES.items():
    filas = verificar_slo(inf.resumen())
    estado = "✅ APRUEBA" if all(f["cumple"] for f in filas) else "❌ NO APRUEBA"
    detalle = " · ".join(f"{f['metrica']} {f['valor']:.0%} {'✓' if f['cumple'] else '✗'}" for f in filas)
    print(f"{estado:<14}{nombre:<26}{detalle}")

## Mitigación 3 · El umbral de similitud (y por qué no basta)

Un **umbral** responde "no sé" sin siquiera llamar al LLM cuando el mejor fragmento no se parece a la pregunta. Ahorra dinero, pero ¿separa las preguntas con respuesta de las que no? Veamos los puntajes del mejor fragmento (gratis):

In [ ]:
pm = puntajes_mejor_fragmento(base, GOLDEN)
for grupo, lista in pm.items():
    print(f"{grupo:<18}", " ".join(f"{s:.2f}" for _, s in sorted(lista, key=lambda x: x[1])))
mejor_ajena = max(s for i, s in pm["fuera_de_alcance"] if i == "F04")
peor_con = min(s for _, s in pm["con_respuesta"])
print(f"\nLa pregunta AJENA saca {mejor_ajena:.2f}: un umbral la descarta.")
print(f"Las preguntas con respuesta llegan a sacar {peor_con:.2f}; las fuera-de-alcance PARECIDAS (F01 a F03) sacan puntajes dentro del mismo rango.")

El umbral descarta lo **ajeno**, pero las preguntas **parecidas pero ausentes** (televisión por cable, el plan de 1 Gbps) puntúan igual que las legítimas: ningún umbral las separa sin perder preguntas buenas. Para esas hace falta la **instrucción de "no sé" en el prompt (v3)** y **medirla**. Mitigar en capas, otra vez.

---
# Parte 11 · El juez LLM (*LLM-as-judge*) y los marcos de evaluación

Las comprobaciones por código no entienden el lenguaje: no saben que "no es normal" equivale a "es un fraude". Para eso se usa **otro modelo como juez**: le das una **rúbrica**, la pregunta, la respuesta y el contexto (o la respuesta esperada), y devuelve un **veredicto estructurado**.

| Juez | Pregunta | Recibe |
| --- | --- | --- |
| **Fidelidad** (*groundedness*) | ¿Todo lo que afirma la respuesta está en el contexto? | Contexto + respuesta |
| **Corrección** | ¿Coincide con la respuesta esperada? | Respuesta esperada + respuesta |

Buenas prácticas que ya conoces de la Sesión 3: **un criterio por juez**, rúbrica explícita, **salida estructurada validada** (Pydantic) y prompt **versionado**.

## Pero un juez es un instrumento... y hay que calibrarlo

Un medidor sin certificar no es un medidor. Los jueces LLM tienen **sesgos conocidos**:

| Sesgo | Qué hace |
| --- | --- |
| **Indulgencia** | Aprueba casi todo (el error más caro: deja pasar las alucinaciones) |
| **Severidad** | Rechaza de más (te hace perder tiempo revisando respuestas buenas) |
| **Sesgo de longitud** | Prefiere respuestas largas y detalladas, aunque un dato esté mal |
| **Autopreferencia** | Favorece respuestas escritas por un modelo parecido a él (por eso el juez ideal es **otro** modelo) |
| **Sensibilidad al prompt** | Cambia el veredicto al cambiar una frase de la rúbrica |

**La calibración** consiste en pasar al juez respuestas que **una persona ya etiquetó** y medir cuánto coinciden. Escribimos 21 respuestas con su etiqueta humana en `datos/etiquetas_juez.json`, incluidas trampas: una cifra equivocada que sí está en el contexto, una extrapolación disfrazada de prudencia, una respuesta larga con un error escondido.

In [ ]:
ETIQUETAS = json.loads(Path("datos/etiquetas_juez.json").read_text(encoding="utf-8"))
por_id = {p["id"]: p for p in GOLDEN}
for e in [x for x in ETIQUETAS if x["id"] in ("J02", "J07", "J14", "J16")]:
    print(f"{e['id']} · {por_id[e['pregunta_id']]['pregunta']}")
    print(f"   respuesta: {e['respuesta']}")
    print(f"   etiqueta humana → fiel: {e['fiel']} · correcta: {e['correcta']}  ({e['nota']})\n")
print(f"{len(ETIQUETAS)} etiquetas · {sum(e['fiel'] for e in ETIQUETAS)} fieles / {sum(not e['fiel'] for e in ETIQUETAS)} no fieles")

### Calibrar el juez

> 📖 **Sección opcional.** En clase se resume en dos cifras (un juez de 3B apenas supera el azar; uno de 12B coincidió casi siempre); las celdas de calibración puedes ejecutarlas después o dejarlas como lectura.

Medimos tres jueces con el mismo modelo, que es el que tengas configurado (`JUEZ_PROVIDER`/`JUEZ_MODEL`): el de **fidelidad v1** (sin rúbrica), el de **fidelidad v2** (rúbrica estricta) y el de **corrección v1**. Las medidas:

- **Exactitud:** fracción de veredictos que coinciden con la persona.
- **Kappa de Cohen:** acuerdo **descontando el azar** (1 = perfecto, 0 = tanto como lanzar una moneda). La exactitud sola engaña cuando las clases están desbalanceadas.
- **Indulgencia:** de lo que la persona rechazó, cuánto aprobó el juez. **Severidad:** de lo que la persona aprobó, cuánto rechazó el juez.

💰 **FYI:** 21 llamadas por juez (63 en total; ≈ US$ 0.03 con un modelo de proveedor; gratis con el modelo local).

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.03 con un modelo de proveedor; gratis con el modelo local)
def mostrar_calibracion(nombre, c):
    print(f"{nombre:<20} exactitud {c['exactitud']:.0%} · kappa {c['kappa']:.2f} · "
          f"indulgencia {c['indulgencia']:.0%} · severidad {c['severidad']:.0%} · sin veredicto {c['sin_veredicto']} · {c['segundos']:.0f} s")
    print(f"{'':<20} el juez aprobó bien {c['aprobo_bien']} · aprobó MAL {c['aprobo_mal']} · rechazó MAL {c['rechazo_mal']} · rechazó bien {c['rechazo_bien']}")

if PUEDE_GASTAR:
    calibraciones = {}
    for nombre, criterio, version in (("fidelidad v1", "fidelidad", "v1"), ("fidelidad v2", "fidelidad", "v2"), ("corrección v1", "correccion", "v1")):
        calibraciones[nombre] = calibrar_juez(ETIQUETAS, GOLDEN, MANUAL, criterio, version)
        mostrar_calibracion(nombre, calibraciones[nombre])
else:
    print("⚪ Celda de pago desactivada.")

In [ ]:
if PUEDE_GASTAR:
    c = calibraciones["fidelidad v2"]
    print("Casos donde el juez de fidelidad v2 NO coincidió con la persona:\n")
    for f in c["filas"]:
        if f["humano"] != f["juez"]:
            print(f"  {f['id']}: persona={'fiel' if f['humano'] else 'no fiel'} · juez={'fiel' if f['juez'] else 'no fiel'}  ← {f['nota']}")

### Lo que enseña esta calibración

1. **El prompt del juez importa tanto como el del RAG.** Pasar de v1 (sin rúbrica) a v2 (rúbrica estricta) cambia el *tipo* de error: de un juez que **rechaza de más** a uno que deja pasar alucinaciones. Ninguno es bueno si el kappa es bajo.
2. **Un modelo pequeño es un mal juez.** Con un juez de 3B parámetros el acuerdo con las personas es **poco mejor que el azar**. Aprueba cifras equivocadas y extrapolaciones, y se confunde con las respuestas largas. Si tu juez es débil, **tu evaluación es decoración**.
3. **Qué hacer entonces:**
   - Usa un modelo **más capaz** que el generador y, si puedes, **distinto** (evita la autopreferencia).
   - **Calibra siempre** contra etiquetas humanas; fija una meta (p. ej. kappa ≥ 0.6) antes de confiar.
   - Usa el juez para **priorizar** qué revisa una persona, no para reemplazarla.
   - **Combina** el juez con las comprobaciones por código: se compensan.
   - Mantén el golden set y las etiquetas **versionados**: son tus pruebas de regresión.

In [ ]:
# Opcional · ¿Cambia el cuadro con un juez más grande?  Un modelo local más capaz, o un proveedor.
# 💰 FYI: con un modelo de proveedor ≈ US$ 0.01 (21 llamadas); con un modelo local grande tarda varios minutos.
MODELO_JUEZ_GRANDE = None      # p. ej. "google/gemma-4-12b-qat" (local) · déjalo en None para omitir esta celda
if PUEDE_GASTAR and MODELO_JUEZ_GRANDE:
    c_grande = calibrar_juez(ETIQUETAS, GOLDEN, MANUAL, "fidelidad", "v2", modelo=MODELO_JUEZ_GRANDE)
    mostrar_calibracion(f"v2 · {MODELO_JUEZ_GRANDE}", c_grande)

> 🔬 **Una referencia.** Con `google/gemma-4-12b-qat` (modelo local de 12B parámetros) el juez de fidelidad v2 coincidió con la persona en **21 de 21** casos (kappa 1.0; en 2 casos no devolvió un JSON válido), con más de un minuto por veredicto en un equipo sin GPU. El mismo prompt de rúbrica, con un modelo más capaz, pasó de azar a acuerdo casi total: **el modelo del juez pesa más que su prompt**. Pero 21 etiquetas son pocas: no lo tomes como garantía; una calibración seria necesita más casos y volver a hacerse al cambiar de modelo o de rúbrica.

### El juez y el chequeo por código, sobre respuestas reales del RAG

Cruzamos las dos herramientas sobre el subconjunto rápido (10 preguntas) con el prompt v3: ¿coinciden el juez de fidelidad y el chequeo de cifras con el diagnóstico?

💰 **FYI:** 10 llamadas del RAG + hasta 10 del juez (≈ US$ 0.01 con un modelo de proveedor; gratis con el modelo local).

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.01 con un modelo de proveedor; gratis con el modelo local)
if PUEDE_GASTAR:
    rapido = [p for p in GOLDEN if p["rapido"]]
    inf_juez = evaluar_rag(base, rapido, k=4, version="v3", juez="fidelidad")
    print(f"{'id':<6}{'diagnóstico':<28}{'cifras sin respaldo':<22}{'juez de fidelidad'}")
    for f in inf_juez.filas:
        print(f"{f.id:<6}{f.diagnostico:<28}{str(f.numeros_sin_respaldo or '—'):<22}{'fiel' if f.fiel else ('NO fiel' if f.fiel is False else 'sin veredicto')}")
    print(f"\nFidelidad según el juez: {inf_juez.resumen().get('fidelidad_juez', 0):.0%}")

### Marcos de evaluación del ecosistema: cuándo usarlos

Hoy medimos **a mano** (con `evaluacion.py`) para que entiendas **qué** mide cada número. En proyectos reales muchos equipos adoptan un marco que ya trae métricas, juez y paneles:

| Marco | Qué ofrece (según su documentación) | Cuándo mirarlo |
| --- | --- | --- |
| **RAGAS** | Un conjunto de métricas para evaluar RAG (p. ej. fidelidad, relevancia de la respuesta, precisión y recall del contexto), varias calculadas con un LLM como juez | Cuando quieres métricas estándar sin escribirlas |
| **TruLens** | Instrumenta la aplicación y evalúa con "funciones de retroalimentación"; popularizó la "tríada del RAG" (relevancia del contexto, fundamentación y relevancia de la respuesta) | Cuando quieres evaluar y ver trazas en un panel |
| **DeepEval** | Pruebas de LLM al estilo de `pytest`, con métricas para RAG | Cuando quieres meter la evaluación en CI |
| **Phoenix** (Arize) | Observabilidad (trazas) y evaluaciones, apoyado en OpenTelemetry | Cuando quieres ver lo que pasa en producción |

**Por qué en el curso no los usamos todavía.** (1) Por dentro, cada métrica es **un conteo o un prompt de juez** como los que escribimos hoy; si no sabes qué mide, no sabes cuándo miente. (2) Muchas usan un **LLM como juez**: heredan lo que vimos en esta parte (se calibra, tiene sesgos). (3) Cambian rápido: **consulta su documentación vigente** antes de adoptar uno. (4) No añadimos dependencias que no necesitas para aprender.

**Cuándo sí.** Cuando ya entiendes tus métricas y necesitas **estandarizar**, **graficar en un panel**, **correr en CI** o **comparar con otros equipos**. Un buen criterio: usa el marco para lo que ya sabrías hacer a mano, y valida un par de casos a mano contra sus resultados.

---
# Parte 12 · Del notebook a producción: caché, costo, actualización y observabilidad

Medimos, mejoramos y evaluamos. Falta lo que decide si el RAG **sobrevive** en producción: que sea **rápido y barato** (caché, latencia, costo), que **siga correcto cuando cambian los documentos** (actualización incremental, ciclo de vida del índice) y que puedas **ver qué pasa** (observabilidad). Todas las celdas de esta parte son pequeñas y, salvo una, **gratis**.

## Cinco ideas

**🧠 Qué es cada una.**

| Idea | En una frase | Dónde se ve el beneficio |
| --- | --- | --- |
| **Caché semántica** | Si ya respondí algo **muy parecido**, devuelvo esa respuesta sin llamar al LLM | Costo y latencia |
| **Caché de prompt / prefijo** | El proveedor reutiliza el procesamiento de la **parte inicial** de un prompt que se repite | Costo y latencia de cada llamada |
| **Actualización incremental** | Cuando cambia un documento, **re-vectorizo solo lo que cambió** (por hash de contenido) | Costo de indexar; que el índice no quede viejo |
| **Ciclo de vida del índice** | El índice tiene **versión**: cambiar de modelo o de fragmentado significa construir otro, probarlo con el golden set y cambiar de uno al otro | Seguridad al desplegar |
| **Observabilidad** | Registrar **cada paso** de cada pregunta para depurar y medir | Saber qué falló y dónde |

**⚙️ Qué tipos de caché hay.**

| Caché | Qué guarda | Cuándo acierta | Riesgo principal |
| --- | --- | --- | --- |
| **Exacta** | Respuesta por texto de pregunta idéntico | Preguntas repetidas al carácter | Casi nunca acierta con lenguaje natural |
| **Semántica** | Respuesta por pregunta **parecida** (embedding + **umbral**) | Paráfrasis | **Falso acierto**: una pregunta casi igual pero con **otra** respuesta recibe la respuesta equivocada con total seguridad |
| **De prompt / prefijo** | El inicio del prompt ya procesado, en el proveedor | Prompts que comparten el mismo comienzo | Hay que **poner lo estable primero y lo variable al final** |
| **De embeddings** | El vector de textos ya vectorizados | Reindexar o repetir consultas | Memoria |
| **De recuperación** | Los fragmentos devueltos para una consulta | Consultas repetidas | Datos **obsoletos** si el documento cambió |

**🎯 Para qué sirve.** En un servicio real las preguntas se repiten mucho (horarios, precios, "¿cómo cancelo?"). Responder esas desde una caché cuesta **milisegundos y cero dólares** en lugar de segundos y tokens. Pero cada caché es una **copia** de la verdad: puede quedar vieja.

**🏗️ Analogía de infraestructura.** Es exactamente lo que ya conoces: un **CDN** o un Redis delante de la API. Y con las mismas reglas: **TTL** (cuánto vive una entrada), **invalidación** (borrar la entrada cuando el origen cambia), **clave de caché** (qué define "la misma petición") y miedo a servir contenido viejo. El actualizador incremental es un **`rsync` frente a un `scp` completo**; el ciclo de vida del índice es un **despliegue azul/verde**; las trazas son tus **logs y métricas** de siempre.

**✏️ Ejemplo a mano (caché semántica).** La respuesta "El plan Familiar cuesta 39 dólares" se guardó para "¿Cuánto cuesta el plan de 300 Mbps?". Llega "¿Cuál es el precio mensual del plan de 300 Mbps?": similitud 0.93 ≥ umbral 0.90 → **acierto**, sin llamar al LLM. Llega "¿Cuánto cuesta el plan de 600 Mbps?": parecidísima de palabras, pero la respuesta correcta es **otra**. Si el umbral es demasiado bajo, la caché responde 39 y **está mal**.

**⚠️ Cuándo no.** No cachees respuestas que dependan de **quién** pregunta (permisos, cuenta del cliente) ni de datos que cambian en minutos. La caché semántica es **la más peligrosa**: úsala con umbral alto y solo para preguntas generales y estables. Y mide la **tasa de falsos aciertos**, no solo la de aciertos.

**🐞 Errores típicos.** (1) Umbral elegido "a ojo". (2) No incluir la **versión del índice y del prompt** en la clave: al cambiarlos, la caché sigue sirviendo respuestas del sistema anterior. (3) Sin TTL ni invalidación. (4) Poner la pregunta del usuario **al inicio** del prompt y arruinar la caché de prefijo. (5) Reindexar todo "por si acaso". (6) Trazar la pregunta completa con datos personales sin pensar en privacidad.

### 1. Caché semántica: ¿qué umbral? (gratis)

Guardamos las 24 preguntas del golden set (con su respuesta esperada como "respuesta guardada") y las atacamos con tres tipos de consulta de `datos/cache_semantica.json`: **12 paráfrasis** (deben acertar y devolver su misma pregunta), **8 casi iguales con otra respuesta** (deben fallar) y **4 ajenas** (deben fallar).

In [ ]:
CACHE_DATOS = json.loads(Path("datos/cache_semantica.json").read_text(encoding="utf-8"))
cache = CacheSemantica(umbral=0.90)
cache.guardar_varias([p["pregunta"] for p in GOLDEN], [p["respuesta_esperada"] for p in GOLDEN])
posicion_de = {p["id"]: i for i, p in enumerate(GOLDEN)}

def similitudes(lista):
    # Para cada consulta: (entrada de la lista, índice de la pregunta guardada más parecida, similitud).
    return [(e, *cache.consultar(e["texto"])[1:]) for e in lista]

parafrasis, casi_iguales, ajenas = (similitudes(CACHE_DATOS[c]) for c in ("parafrasis", "casi_iguales", "ajenas"))
print(f"{'umbral':>7}{'aciertos':>12}{'(bien)':>9}{'falsos: casi iguales':>24}{'falsos: ajenas':>17}")
for umbral in (0.70, 0.75, 0.80, 0.85, 0.90, 0.95):
    bien = sum(s >= umbral and i == posicion_de[e["id"]] for e, i, s in parafrasis)
    hit = sum(s >= umbral for _, _, s in parafrasis)
    print(f"{umbral:>7.2f}{hit:>9}/{len(parafrasis)}{bien:>7}/{len(parafrasis)}"
          f"{sum(s >= umbral for _, _, s in casi_iguales):>20}/{len(casi_iguales)}{sum(s >= umbral for _, _, s in ajenas):>14}/{len(ajenas)}")

print("\nLas consultas casi iguales que MÁS se parecen a una pregunta guardada (¡y piden otra cosa!):")
for e, i, s in sorted(casi_iguales, key=lambda x: -x[2])[:3]:
    print(f"  {s:.2f}  «{e['texto']}»\n        se parece a «{GOLDEN[i]['pregunta']}»")

### Cómo leer la tabla

- **El umbral es un compromiso medible:** bajo → más aciertos y también **falsos aciertos** (respuesta equivocada servida con seguridad); alto → casi no acierta. Con este modelo de embeddings, las paráfrasis puntúan en un rango amplio y una pregunta **casi igual pero distinta** puede puntuar **más** que una paráfrasis legítima. Ningún umbral las separa del todo: por eso la caché semántica se usa con umbral alto, para preguntas generales y estables, y con **revisión** de los falsos aciertos.
- **El umbral depende del modelo de embeddings:** los valores de similitud de otro modelo caen en otro rango. Se calibra **con tus preguntas**, no se copia.
- Una **tasa de aciertos** alta es buena noticia solo si la **tasa de falsos aciertos** es baja. Mide las dos.

### 2. Caché de prompt (prefijo): el orden importa (gratis)

Los proveedores pueden reutilizar el procesamiento de la **parte inicial** de un prompt que se repite entre llamadas (según su documentación, con condiciones y precios propios: consúltalos). Para que funcione, **lo que no cambia va primero y lo que cambia va al final**. Medimos cuánto prefijo comparten dos prompts reales con el orden del curso (reglas → contexto → pregunta) y con el orden invertido:

In [ ]:
pregs = [GOLDEN[0]["pregunta"], GOLDEN[2]["pregunta"]]
prompts = [construir_prompt(q, recuperar(base, q, k=4), "v3") for q in pregs]
invertidos = [f"Pregunta del cliente: {q}\n\n" + p.rsplit("Pregunta del cliente:", 1)[0] for q, p in zip(pregs, prompts)]

for etiqueta, par in (("reglas → contexto → pregunta (el del curso)", prompts), ("pregunta primero (invertido)", invertidos)):
    comun = os.path.commonprefix(par)
    print(f"{etiqueta:<46} prefijo común: {len(comun.split()):>3} de {len(par[0].split())} palabras ({len(comun.split()) / len(par[0].split()):.0%})")
print("\nEn el prompt del curso, lo común son las reglas; el contexto cambia con cada pregunta. "
      "Si el contexto fuera idéntico (como el documento completo del contextual retrieval), todo el prefijo sería reutilizable.")

### 3. Latencia y costo: ¿dónde se va el tiempo? Y qué trazar (💰 o gratis con modelo local)

Un RAG tiene etapas, y cada una tiene su tiempo: **embedding de la consulta → búsqueda → (reranking) → armar el prompt → generación**. Casi siempre **la generación domina**; por eso la caché que evita la generación es la que más ahorra. Medimos una pregunta de verdad y dejamos lo que **debería quedar registrado** de cada petición:

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.0005 con un modelo de proveedor: 1 pregunta; gratis con el modelo local)
def responder_con_traza(pregunta, k=4, version="v3"):
    tr = Traza(datos={"pregunta": pregunta, "version_prompt": version, "modelo_generacion": f"{GEN}", "modelo_embeddings": modelo_embeddings(),
                      "version_indice": "manual_soporte · oraciones 70 · k=4", "k": k})
    with tr.etapa("embedding_consulta"):
        vector = vector_consulta(pregunta)
    with tr.etapa("busqueda"):
        top = base.buscar(vector, k=k)
    with tr.etapa("prompt"):
        prompt = construir_prompt(pregunta, top, version)
    with tr.etapa("generacion"):
        r = generar(prompt, max_tokens=300)
    citas = citas_en(r.texto)
    tr.datos.update({
        "fragmentos": [{"id": f.id, "puntaje": round(f.puntaje, 3)} for f in top],
        "tokens_entrada": r.tokens_entrada, "tokens_salida": r.tokens_salida, "costo_usd": round(r.costo, 6),
        "citas": sorted(citas), "citas_validas": all(1 <= c <= len(top) for c in citas) and bool(citas),
        "cache": "fallo",
    })
    return tr, r

if PUEDE_GASTAR:
    tr, r = responder_con_traza(GOLDEN[4]["pregunta"])
    gasto.registrar(r)
    print(tr)
    ms = tr.tiempos_ms
    tiempos_cache = []
    for _ in range(5):                                            # mediana de 5 consultas a la caché (la primera puede incluir una carga)
        t0 = time.perf_counter(); cache.consultar(GOLDEN[4]["pregunta"] + " ¿verdad?"); tiempos_cache.append((time.perf_counter() - t0) * 1000)
    t_cache = sorted(tiempos_cache)[2]
    print(f"\nDe los {ms['embedding_consulta'] + ms['busqueda'] + ms['prompt'] + ms['generacion']:,} ms, la generación fue {ms['generacion'] / sum(ms.values()):.0%}. "
          f"Un acierto de caché semántica cuesta solo el embedding de la pregunta y la comparación: {t_cache:.0f} ms.")
else:
    print("⚪ Celda de pago desactivada.")

**Qué trazar en cada petición** (lo que ves en el JSON de arriba, y lo que falta):

| Grupo | Qué registrar | Para qué |
| --- | --- | --- |
| **Identidad** | id de la petición, usuario o sesión (anonimizados), fecha | Reproducir un caso |
| **Versiones** | prompt, modelo, modelo de embeddings, **versión del índice**, parámetros (`k`, umbral) | Saber qué sistema respondió |
| **Recuperación** | ids y puntajes de los fragmentos, si hubo caché | Diagnosticar la capa |
| **Generación** | tokens de entrada y salida, costo, motivo de parada | Costo y recortes |
| **Tiempos** | por etapa, no solo el total (y mira el **p95**, no el promedio) | Dónde optimizar |
| **Calidad** | citas válidas, "no sé", veredicto del juez en una **muestra**, voto del usuario | Detectar regresiones |
| **Privacidad** | qué datos personales guardas y por cuánto tiempo | Cumplimiento |

Herramientas reales para esto (sin instalarlas hoy): trazado con **OpenTelemetry** y plataformas como **Langfuse** o **Phoenix**; la idea es la misma que ya aplicas con tus logs y métricas.

### 4. Actualización incremental: reindexar solo lo que cambió (gratis)

Cada fragmento se identifica por el **hash de su contenido**. Al sincronizar, se vectorizan solo los fragmentos cuyo hash es nuevo y se descartan los que ya no existen. Probamos con el manual: sin cambios, con un cambio de precio y con una sección nueva.

In [ ]:
import tempfile

carpeta_tmp = Path(tempfile.mkdtemp())
doc_tmp = carpeta_tmp / "manual.md"
indice = IndiceIncremental(crear_almacen("memoria"))

def sincronizar(descripcion, texto):
    doc_tmp.write_text(texto, encoding="utf-8")
    r = indice.sincronizar(fragmentar(leer_documento(doc_tmp), "oraciones", 70, 1))
    ahorro = 1 - r["palabras_vectorizadas"] / r["palabras_totales"]
    print(f"{descripcion:<30} fragmentos {r['total']:>2} · vectorizados {r['vectorizados']:>2} · reutilizados {r['reutilizados']:>2} · "
          f"borrados {r['borrados']} · ahorro {ahorro:>4.0%} de las palabras a vectorizar")

sincronizar("1. Primera indexación", documento_manual)
sincronizar("2. Sin cambios", documento_manual)
precio_nuevo = documento_manual.replace("cuesta 39 dólares al mes", "cuesta 42 dólares al mes")
sincronizar("3. Cambia un precio (39→42)", precio_nuevo)
con_seccion = precio_nuevo.rstrip() + "\n\n## 10. Horario en feriados\n\nEn feriados nacionales el soporte telefónico atiende de 10:00 a 14:00.\n"
sincronizar("4. Se añade una sección", con_seccion)

mejor = indice.almacen.buscar(vector_consulta("¿Cuánto cuesta el plan de 300 Mbps?"), k=1)[0]
print(f"\nLa búsqueda ya ve el cambio: «…{mejor.texto[mejor.texto.find('Familiar'):][:75]}…»")

### Cómo leer este resultado

- Un cambio de precio **re-vectoriza 1 fragmento** (y descarta el viejo); el resto se **reutiliza**. En un corpus de miles de fragmentos, la diferencia entre reindexar todo y reindexar lo que cambió es la que hace posible mantener el RAG al día.
- Lo mismo vale para el **grafo** (Parte 6): reextraer solo las secciones que cambiaron.
- **Lo que esta demo no resuelve:** los fragmentos de un documento **borrado** hay que eliminarlos (con un almacén real, por id o por metadato), la **caché semántica** y la de recuperación pueden seguir sirviendo la respuesta vieja hasta que **las invalides** (por TTL o al cambiar el documento), y si cambias de **modelo de embeddings** o de **tamaño de fragmento**, todos los hashes valen y todo se reindexa.

### Ciclo de vida del índice

1. **Versiona** el índice: modelo de embeddings, tamaño y estrategia de fragmentado, esquema de metadatos. Esa versión viaja en cada **traza** y en la **clave de la caché**.
2. **Un cambio estructural = un índice nuevo.** Constrúyelo al lado del actual (como un despliegue azul/verde), pásale el **golden set** y el SLO, y solo entonces cambia el tráfico. Conserva el anterior para volver atrás.
3. **Vigencia y retiros:** guarda metadatos de vigencia (Parte 10) y borra de verdad lo que se retira (derecho al olvido, documentos derogados).
4. **Vigila la deriva:** una muestra de preguntas reales de producción pasa periódicamente por el golden set y por el juez; si la calidad cae, algo cambió (documentos, modelo, tráfico).

> 🧰 **Regla práctica:** la mejor caché es la que **sabes invalidar**. Antes de cachear algo, responde: ¿cuándo deja de ser verdad?

---
# Parte 13 · Práctica integrada: el Medidor de calidad RAG

La práctica de la sesión es [`practica/medidor_calidad_rag.py`](practica/medidor_calidad_rag.py): un programa único y comentado que **evalúa tu RAG con el golden set** y entrega un **informe** (consola, JSON y Markdown) con Recall@K, Precision@K, MRR, las métricas de generación, el diagnóstico por capa y si **cumple el SLO**. Si no lo cumple, termina con código de salida 1: sirve como **puerta de calidad en CI**.

```bash
# Desde la carpeta s07-rag-avanzado/
uv run python practica/medidor_calidad_rag.py --sin-generacion                       # solo recuperación (gratis, segundos)
uv run python practica/medidor_calidad_rag.py --rapido --version v3                  # las 10 preguntas rápidas
uv run python practica/medidor_calidad_rag.py --tecnica small-to-big -k 2 --version v3
uv run python practica/medidor_calidad_rag.py --version v3 --juez fidelidad          # añade el juez
uv run python practica/medidor_calidad_rag.py --version v3 --comparar-con informes/anterior.json
uv run python practica/medidor_calidad_rag.py --version v3 --exigir-slo              # código de salida 1 si no cumple
```

Aquí hacemos lo mismo, dentro del notebook, con el subconjunto rápido:

💰 **FYI:** 10 llamadas al LLM (≈ US$ 0.005 con un modelo de proveedor; gratis con el modelo local).

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.005 con un modelo de proveedor; gratis con el modelo local)
if PUEDE_GASTAR:
    rapido = [p for p in GOLDEN if p["rapido"]]
    informe = evaluar_rag(base, rapido, k=4, version="v3")
    imprimir_resumen(informe, "Medidor de calidad RAG · set rápido")
    print()
    for f in verificar_slo(informe.resumen()):
        print(f"  {'✅' if f['cumple'] else '❌'} {f['metrica']:<22} {f['valor']:>5.0%}   (regla {f['regla']})")
    ruta_json, ruta_md = guardar_informe(informe, "informes", "notebook_set_rapido")
    print(f"\nInforme guardado en {ruta_json} y {ruta_md}")
else:
    print("⚪ Celda de pago desactivada.")

In [ ]:
if PUEDE_GASTAR:
    print(Path("informes/notebook_set_rapido.md").read_text(encoding="utf-8"))

## 🏁 Retos

1. **Tu propio golden set.** Escribe 10 preguntas sobre un documento tuyo (un reglamento, un manual del trabajo) con `secciones` y `claves`, e inclúyelas con `--golden`. ¿Cuántas fallan en recuperación y cuántas en generación?
2. **Una técnica, una falla.** Elige **una** falla del informe (p. ej. las abstenciones indebidas), aplica **una** técnica y vuelve a medir. Escribe en dos líneas qué cambió y si fue señal o ruido.
3. **Prompt v4.** Crea `prompts/rag_respuesta/v4.j2`, anótalo en `prompts/CHANGELOG.md` y compáralo con `v3` usando `--version v4 --comparar-con`.
4. **Rompe tu juez.** Escribe 3 respuestas nuevas (con un error sutil) para `datos/etiquetas_juez.json` y comprueba si tu juez las atrapa.
5. **Mide tu grafo.** Añade 3 servicios al catálogo (`datos/servicios_ti.md`) y sus tripletas curadas; ¿la extracción con el LLM las encuentra?
6. **Pregunta global.** Escribe una pregunta que **solo** un grafo pueda responder bien, y otra que **solo** un vectorial. Explica por qué.
7. **Tu propio audio.** Transcribe una grabación tuya (una reunión, una clase) a `datos/media/`, escribe 6 preguntas con el minuto esperado y mide hit@1 y hit@3. ¿Qué términos técnicos se escribieron mal?
8. **Matryoshka con tus datos.** Repite la tabla de la Parte 7 con tu propio documento y tu propio golden set: ¿hasta cuántas dimensiones puedes bajar sin perder recall?
9. **Tu umbral de caché.** Escribe 10 paráfrasis y 5 preguntas casi iguales con otra respuesta sobre tus datos, y elige el umbral que defenderías. ¿Qué falso acierto te preocupa más?

---
# Parte 14 · Cierre

## Lo que aprendiste

**Técnicas avanzadas de recuperación**
1. Un RAG básico falla con preguntas **multi-sección y multi-salto**; subir *k* sube Recall pero hunde Precision.
2. ***Small-to-big***: buscar con fragmentos pequeños, entregar el contexto grande; los **encabezados contextuales** (y el *contextual retrieval*, con un LLM) dan tema a fragmentos sueltos, y **solo ayudan donde se mide que ayudan**.
3. **Descomposición** (separar en subpreguntas, buscar y fusionar con RRF) ataca lo multi-salto; **HyDE** es una apuesta que hay que medir.
4. **GraphRAG** = grafo de conocimiento + RAG: brilla en preguntas **relacionales y globales**, cuesta **extraer, validar y mantener** el grafo, y **no** conviene si tu corpus es pequeño o tus preguntas son directas. Microsoft GraphRAG y LightRAG son las implementaciones de referencia.
5. Hay **recuperadores modernos**: dispersos aprendidos (SPLADE) y de interacción tardía (ColBERT; ColPali para páginas como imagen). Los **embeddings** se optimizan con **Matryoshka** y **cuantización**, y se decide **midiendo**.
6. El **RAG multimodal** de audio y video es un RAG de texto con un parser nuevo (ASR) que **conserva los tiempos**; la respuesta cita `mm:ss`; la transcripción se **revisa**; y en video la voz sostiene el RAG.

**Evaluación y alucinaciones**
7. Un RAG se evalúa **por capas**: recuperación (Recall@K, Precision@K, MRR) y generación (dato correcto, citas, "no sé", cifras con respaldo), con un **golden set** versionado.
8. El **diagnóstico por capa** dice *dónde* actuar; **fidelidad ≠ corrección**.
9. Las alucinaciones tienen **causas identificables** (prompt sin "no sé", contexto obsoleto, recuperación pobre, extrapolar…) y cada una su mitigación; algunas están **en los datos**, no en el prompt.
10. Un **juez LLM** es un instrumento que se **calibra** contra etiquetas humanas; uno débil es decoración. Los marcos (RAGAS, TruLens, DeepEval, Phoenix) ayudan cuando ya entiendes lo que miden.
11. **Medir → cambiar → volver a medir**, con un **SLO** que decida si el cambio se despliega.

**Producción**
12. La **caché semántica** ahorra costo y latencia, pero el umbral se calibra midiendo los **falsos aciertos**; la de **prompt** exige poner lo estable primero.
13. La **actualización incremental** (hash de contenido) evita reindexar todo; el índice se **versiona** y se cambia como un despliegue azul/verde; y se **traza** cada petición.

## 🧭 Qué técnica para qué falla

| Falla que mides | Técnica | Costo extra |
| --- | --- | --- |
| Falta contexto alrededor del fragmento correcto | *Small-to-big* (sección o ventana) | Más tokens por pregunta |
| Fragmentos genéricos o ambiguos | Encabezados contextuales; *contextual retrieval* | Nada / 1 llamada al LLM por fragmento al indexar |
| La pregunta necesita **dos temas** | Descomposición + RRF | 1 llamada al LLM por pregunta |
| La pregunta no se parece al texto de la respuesta | HyDE (¡mídelo!) | 1 llamada al LLM por pregunta |
| Falta el vocabulario exacto (códigos, nombres) | Búsqueda híbrida (Sesión 6) | Un índice más |
| El orden del top-k es malo | Reranking (Sesión 6) | Un modelo más por pregunta |
| Preguntas **relacionales o globales** | **GraphRAG** | Extraer, validar y mantener el grafo |
| El índice pesa demasiado | Matryoshka e int8 | Verificar con tus preguntas |
| La información está en **audio o video** | Transcripción con tiempos (+ fotogramas) | Cómputo de ASR; revisar la transcripción |
| Documentos **visuales** (tablas, gráficos, diseño) | ColPali / ColQwen (conceptual) | Modelo de visión y mucho almacenamiento |
| Preguntas repetidas, costo y latencia | Caché semántica / de prompt | Falsos aciertos; invalidación |
| El documento cambia | Actualización incremental | Hash y control de versiones |

## ✅ Autoevaluación

1. ¿Por qué Recall@K sube y Precision@K baja al aumentar K? ¿Qué técnica sube Recall sin hundir Precision?
2. ¿Qué dos fallas distintas puede tener una respuesta a una pregunta multi-sección? ¿Cómo sabes cuál ocurrió?
3. ¿Cuándo preferirías *small-to-big* en modo `seccion` y cuándo en modo `ventana`?
4. ¿Por qué la pregunta "si falla X, ¿qué se afecta?" es difícil para un RAG vectorial y fácil para un grafo?
5. Nombra dos costos de GraphRAG que no aparecen en una demo y una situación donde **no** lo usarías.
6. ¿Por qué una arista falsa en el grafo contamina muchas respuestas?
7. ¿Qué diferencia a un bi-encoder, a SPLADE y a ColBERT en lo que guardan por texto? ¿Qué precio paga la interacción tardía?
8. ¿Qué hay que hacer después de truncar un vector (Matryoshka) y qué pasa si solo truncas los fragmentos y no la pregunta?
9. En un RAG de audio, ¿por qué se guardan `inicio` y `fin`? ¿Qué harías para que un término técnico no se transcriba mal?
10. Un RAG saca 0 % de alucinaciones y 40 % de "no sé" indebido. ¿Está mejor o peor que otro con 5 % y 5 %? ¿Qué más mirarías?
11. Explica con un ejemplo la diferencia entre fidelidad y corrección.
12. ¿Qué sesgos tiene un juez LLM y cómo compruebas si el tuyo los tiene?
13. ¿Qué es un falso acierto de la caché semántica? ¿Cómo eliges el umbral?
14. ¿Por qué se pone la pregunta del usuario al final del prompt si quieres aprovechar la caché de prefijo?
15. ¿Qué hace un SLO en un flujo de CI de un RAG?

## 💰 Resumen de gasto de esta sesión

In [ ]:
total_evaluaciones = sum(i.resumen()["costo_usd"] for i in INFORMES.values())
print(f"Llamadas sueltas: {len(gasto.resultados)} · US$ {gasto.total:.6f}")
print(f"Evaluaciones completas ({len(INFORMES)} informes): US$ {total_evaluaciones:.6f}")
print("(Con el modelo local el costo es 0. Los jueces y las extracciones de tripletas no se suman aquí: súmalos a mano si usas un proveedor.)")

## 📖 Glosario

| Término | Definición |
| --- | --- |
| **Golden set** | Preguntas con respuesta correcta escrita por una persona; la vara de medir |
| **SLI / SLO** | Indicador que se mide / mínimo aceptable que se exige |
| **Recall@K** | Secciones relevantes en el top-K ÷ secciones relevantes que existen |
| **Precision@K** | Fragmentos relevantes del top-K ÷ K |
| **MRR** | 1 ÷ posición del primer resultado relevante |
| **Multi-salto** | Pregunta que exige encadenar datos de varios lugares (y a veces calcular) |
| ***Small-to-big* / *parent-document*** | Buscar con fragmentos pequeños y entregar el contexto grande al que pertenecen |
| **Encabezado contextual** | Anteponer sección o título al fragmento antes de vectorizarlo |
| **Descomposición** | Separar una pregunta en subpreguntas de un solo tema |
| **RRF** | Fusión por rango recíproco: combina varias listas sin comparar sus puntajes |
| **HyDE** | Buscar con una respuesta hipotética escrita por el LLM |
| **Grafo de conocimiento** | Red de entidades (nodos) y relaciones (aristas) |
| **Tripleta** | Un hecho `(sujeto, relación, objeto)` |
| **GraphRAG** | RAG que usa un grafo de conocimiento (búsqueda local y global) |
| **Búsqueda local / global** | Entidades de la pregunta y sus vecinos / resúmenes por comunidad del grafo |
| **Comunidad** | Grupo de nodos muy conectados entre sí (algoritmo de Leiden) |
| **Fidelidad (*groundedness*)** | Todo lo afirmado está respaldado por el contexto |
| **Corrección** | Lo afirmado es verdad (coincide con lo esperado) |
| **Alucinación** | Respuesta plausible pero no respaldada o falsa |
| **Abstención** | Responder "no sé" cuando el contexto no alcanza |
| **LLM-as-judge** | Usar un modelo, con rúbrica, para evaluar respuestas |
| **Kappa de Cohen** | Acuerdo entre evaluadores descontando el azar |
| **Regresión** | Que un cambio empeore algo que ya funcionaba; se detecta volviendo a medir |
| **Contextual retrieval** | Un LLM escribe una frase de contexto por fragmento antes de vectorizarlo |
| **SPLADE** | Recuperación dispersa aprendida: pesos por palabra, con expansión a palabras relacionadas |
| **Interacción tardía (*late interaction*)** | Un vector por token y comparación palabra a palabra (MaxSim); ColBERT |
| **MaxSim** | Para cada token de la pregunta, su mejor similitud en el documento; se suman |
| **ColPali / ColQwen** | Interacción tardía sobre la imagen de una página, sin OCR |
| **Matryoshka (MRL)** | Embeddings entrenados para que los primeros números formen un buen vector corto; se truncan y renormalizan |
| **Cuantización** | Guardar cada número del vector con menos bits (int8, 1 bit) |
| **ASR / Whisper** | Reconocimiento automático de voz; Whisper es un modelo abierto de ASR |
| **Segmento** | Tramo de habla con `inicio`, `fin` y texto, que devuelve el ASR |
| **`initial_prompt`** | Vocabulario que se le da a Whisper para que espere ciertos términos |
| **Caché semántica** | Reutilizar una respuesta cuando la pregunta nueva se parece lo bastante a una guardada (umbral de coseno) |
| **Falso acierto (de caché)** | La caché responde a una pregunta casi igual que pedía otra cosa |
| **Caché de prompt / prefijo** | El proveedor reutiliza el procesamiento del inicio de un prompt que se repite |
| **Actualización incremental** | Re-vectorizar solo los fragmentos cuyo hash de contenido cambió |
| **Traza** | Registro de las etapas, tiempos, versiones y costo de una petición |

## ➡️ En la sesión siguiente
Con un RAG medido y mejorado tienes la base para **dar al modelo herramientas y capacidad de decidir**: los **agentes**. Tu golden set y tu Medidor de calidad serán la prueba de regresión del hito **H3**.